In [ ]:
# @title 00 . session { display-mode: "form" }
# Environment check + shared helpers for the combined H3 + LTX-2.5 Studio.
# Run once per runtime, before anything else.
import getpass, hashlib, json, os, re, shutil, signal, subprocess, sys, threading, time, uuid
import warnings
from pathlib import Path

import psutil
import requests
import torch
from IPython.display import HTML, display

warnings.filterwarnings("ignore", category=DeprecationWarning)
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

GIB = 1024 ** 3
CONTENT = Path("/content")
COMFY_DIR = CONTENT / "ComfyUI"
CUSTOM_NODES = COMFY_DIR / "custom_nodes"
WORK = CONTENT / "h3lab"
MODELS = COMFY_DIR / "models"
OUTPUT_DIR = COMFY_DIR / "output"
COMFY_LOG = WORK / "comfy.log"
WHEEL_DIR = CONTENT / "_wheels"

GGUF_DIR = CUSTOM_NODES / "ComfyUI-GGUF"
TURBO_DIR = CUSTOM_NODES / "ComfyUI-MiniMax-H3-Turbo"
CONTROL_DIR = CUSTOM_NODES / "H3LabControl"
H3_CONTROL_DIR = CONTROL_DIR
LTX_CONTROL_DIR = CUSTOM_NODES / "LTX25LabControl"
FP16SAFE_DIR = CUSTOM_NODES / "ComfyUI-MiniMaxH3-FP16Safe"
SPECTRUM_DIR = CUSTOM_NODES / "ComfyUI-Spectrum-MiniMax-H3"
FP16_FIX_FILE = CUSTOM_NODES / "minimax_h3_fp16_fix.py"

COND_DIR = WORK / "conditioning"
LAT_DIR = WORK / "latents"
LTX_RESULTS = WORK / "ltx_results"

BASE_URL = "http://127.0.0.1:8188"

for _dir in (WORK, WHEEL_DIR, COND_DIR, LAT_DIR, LTX_RESULTS):
    _dir.mkdir(parents=True, exist_ok=True)

H3 = globals().setdefault("H3", {})
LTX = globals().setdefault("LTX", {})


def _box_html(title, rows=None, kind="info", note="", details=""):
    colors = {"ok": "#0d7a3f", "warn": "#9a6b00", "err": "#b3261e",
              "info": "#1a5fb4", "idle": "#5f6368", "run": "#1a5fb4"}
    color = colors.get(kind, colors["info"])
    wrap = "white-space:normal;overflow-wrap:anywhere;word-break:break-word"
    body = ""
    if rows:
        body += ('<table style="border-collapse:collapse;width:100%;'
                 'font:12px/1.55 monospace;' + wrap + '">')
        for label, value in rows:
            body += ('<tr><td style="width:96px;padding:2px 12px 2px 0;color:#5f6368;'
                     'white-space:nowrap;vertical-align:top">' + str(label) + '</td>'
                     '<td style="padding:2px 0;' + wrap + '">' + str(value)
                     + '</td></tr>')
        body += "</table>"
    if note:
        body += ('<div style="margin-top:6px;color:#5f6368;font:12px/1.5 monospace;'
                 + wrap + '">' + note + "</div>")
    if details:
        body += ('<details style="margin-top:6px"><summary style="cursor:pointer;'
                 'color:#5f6368;font:12px monospace">log tail</summary>'
                 '<pre style="max-height:340px;overflow:auto;background:#f6f8fa;'
                 'padding:8px;font:11px/1.45 monospace;margin:6px 0 0">'
                 + details + '</pre></details>')
    return ('<div style="border:1px solid ' + color + ';border-radius:8px;margin:4px 0;'
            'overflow:hidden;max-width:100%;box-sizing:border-box;' + wrap + '">'
            '<div style="background:' + color + ';color:#fff;padding:6px 12px;'
            'font:600 12px/1.4 monospace;' + wrap + '">' + title + '</div>'
            '<div style="padding:8px 12px;' + wrap + '">' + body + '</div></div>')


def box(title, rows=None, kind="info", note="", details=""):
    display(HTML(_box_html(title, rows, kind, note, details)))


class LiveBox:
    """Box that appears immediately and updates in place while work runs."""

    def __init__(self, key, title, rows=None, kind="run", note=""):
        self.display_id = "h3live-%s-%s" % (key, uuid.uuid4().hex[:6])
        self.title, self.rows = title, list(rows or [])
        self.kind, self.note, self.details = kind, note, ""
        self._render(first=True)

    def _render(self, first=False):
        try:
            display(HTML(_box_html(self.title, self.rows, self.kind, self.note,
                                   self.details)),
                    display_id=self.display_id, update=not first)
        except Exception:
            pass

    def set(self, title=None, rows=None, kind=None, note=None, details=None):
        if title is not None:
            self.title = title
        if rows is not None:
            self.rows = list(rows)
        if kind is not None:
            self.kind = kind
        if note is not None:
            self.note = note
        if details is not None:
            self.details = details
        self._render()

    def finish(self, title, rows=None, kind="ok", note="", details=""):
        self.set(title=title, rows=rows, kind=kind, note=note, details=details)


class Ticker:
    """Refresh a LiveBox every interval seconds via a build(live) callback."""

    def __init__(self, live, build, interval=2.5):
        self.live, self.build = live, build
        self.interval = max(0.5, float(interval))
        self.stop = threading.Event()
        self.thread = threading.Thread(target=self._run, daemon=True)

    def _run(self):
        while not self.stop.wait(self.interval):
            try:
                self.build(self.live)
            except Exception:
                pass

    def start(self):
        try:
            self.build(self.live)
        except Exception:
            pass
        self.thread.start()
        return self

    def stop_now(self):
        self.stop.set()


def fmt_secs(seconds):
    seconds = int(max(0, seconds))
    if seconds < 60:
        return "%ds" % seconds
    minutes, sec = divmod(seconds, 60)
    if minutes < 60:
        return "%dm %02ds" % (minutes, sec)
    hours, minutes = divmod(minutes, 60)
    return "%dh %02dm" % (hours, minutes)


def pbar(fraction, width=24):
    try:
        fraction = float(fraction)
    except Exception:
        fraction = 0.0
    fraction = min(1.0, max(0.0, fraction))
    filled = int(round(fraction * width))
    return "[" + "#" * filled + "-" * (width - filled) + "] %3d%%" % int(round(fraction * 100))


def gb(nbytes):
    return "%.2f GiB" % (nbytes / GIB)


def sh(cmd, cwd=None, timeout=3600, env=None):
    base = dict(os.environ, GIT_TERMINAL_PROMPT="0", HF_HUB_DISABLE_IMPLICIT_TOKEN="1")
    if env:
        base.update(env)
    p = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True, timeout=timeout,
                       env=base)
    return p.returncode, (p.stdout or "") + (p.stderr or "")


def tail_comfy_log(offset=0, limit=120):
    if not COMFY_LOG.exists():
        return ""
    text = COMFY_LOG.read_bytes()[offset:].decode("utf-8", errors="replace")
    lines = [l for l in text.splitlines() if l.strip()]
    return "\n".join(lines[-limit:])


ANSI_RE = re.compile(r"\x1b\[[0-9;]*m")
TQDM_RE = re.compile(
    r"(\d+)%\|[^|\n]*\|\s*(\d+)/(\d+)\s*\[([0-9:]+)<([0-9:]+|\?)"
    r"(?:,\s*([0-9.]+)(s/it|it/s))?")


def parse_clock(value):
    total = 0.0
    for part in value.split(":"):
        total = total * 60 + float(part)
    return total


def latest_progress(text):
    matches = list(TQDM_RE.finditer(ANSI_RE.sub("", text).replace("\r", "\n")))
    if not matches:
        return None
    percent, done, total, elapsed, _remaining, rate, unit = matches[-1].groups()
    done, elapsed_s = int(done), parse_clock(elapsed)
    return {"percent": int(percent), "steps_completed": done, "steps_total": int(total),
            "elapsed_in_loop_s": elapsed_s,
            "observed_s_per_step": (elapsed_s / done) if done else None,
            "reported_s_per_step": (float(rate) if unit == "s/it"
                                    else (1.0 / float(rate) if rate and float(rate) else None))
            if rate else None}


def stop_comfy():
    proc, handle = H3.get("proc"), H3.get("log_handle")
    if proc and proc.poll() is None:
        try:
            os.killpg(proc.pid, signal.SIGTERM)
            proc.wait(timeout=20)
        except Exception:
            try:
                os.killpg(proc.pid, signal.SIGKILL)
                proc.wait(timeout=10)
            except Exception:
                pass
    if handle:
        try:
            handle.close()
        except Exception:
            pass
    H3["proc"], H3["log_handle"] = None, None
    H3["server_family"] = None


def start_comfy(dtype_flag, wait_s=1200, extra=()):
    """Start ComfyUI fresh. Returns (system_stats, log_offset_at_start)."""
    stop_comfy()
    handle = COMFY_LOG.open("ab")
    cmd = [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188",
           "--disable-auto-launch", "--preview-method", "none", "--cache-classic"]
    if dtype_flag:
        cmd.append(dtype_flag)
    cmd += list(extra)
    proc = subprocess.Popen(cmd, cwd=str(COMFY_DIR), stdout=handle,
                            stderr=subprocess.STDOUT, start_new_session=True,
                            env=dict(os.environ, PYTHONUNBUFFERED="1"))
    H3["proc"], H3["log_handle"] = proc, handle
    offset = handle.tell()
    started = time.time()
    while time.time() - started < wait_s:
        if proc.poll() is not None:
            tail = tail_comfy_log(max(0, offset - 40000), 30)
            stop_comfy()
            raise RuntimeError("ComfyUI exited during startup:\n" + tail)
        try:
            r = requests.get(BASE_URL + "/system_stats", timeout=3)
            if r.status_code == 200:
                return r.json(), offset
        except Exception:
            pass
        time.sleep(2)
    stop_comfy()
    raise RuntimeError("ComfyUI did not answer /system_stats within %ds" % wait_s)


def get_object_info():
    return requests.get(BASE_URL + "/object_info", timeout=600).json()


def server_up():
    try:
        return requests.get(BASE_URL + "/system_stats", timeout=3).status_code == 200
    except Exception:
        return False


def validate_graph(graph, object_info):
    errors = []
    for node_id, spec in graph.items():
        info = object_info.get(spec["class_type"])
        if info is None:
            errors.append("%s: unknown class %s" % (node_id, spec["class_type"]))
            continue
        schema = info.get("input", {})
        allowed = (set(schema.get("required", {})) | set(schema.get("optional", {}))
                   | set(schema.get("hidden", {})))
        supplied = set(spec["inputs"])
        missing = set(schema.get("required", {})) - supplied
        if missing:
            errors.append("%s/%s missing %s" % (node_id, spec["class_type"], sorted(missing)))
        # Autogrow inputs show up as dotted children ("ref_images.ref_image_0");
        # accept them when their prefix is a declared input.
        unknown = {name for name in supplied - allowed
                   if name.split(".", 1)[0] not in allowed}
        if unknown:
            errors.append("%s/%s unknown %s" % (node_id, spec["class_type"], sorted(unknown)))
    return errors


_host = psutil.virtual_memory()
_disk = shutil.disk_usage(str(CONTENT))
box("00 . session ready", [
    ("python", sys.version.split()[0]),
    ("torch", "%s  (cuda %s)" % (torch.__version__, torch.version.cuda)),
    ("gpu", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU"),
    ("host RAM", "%.1f GiB total / %.1f GiB free" % (_host.total / GIB, _host.available / GIB)),
    ("/content free", gb(_disk.free)),
    ("models", "MiniMax H3 + LTX-2.5"),
], kind="ok",
   note="Next: 01 base stack, 02 models, then 03 launches the combined Studio. The UI "
        "switches between H3 and LTX-2.5; the notebook only sets up and serves.")


In [ ]:
# @title 01 . base stack { display-mode: "form" }
# ComfyUI 0.33.0 + ComfyUI-GGUF at pinned commits, patched for both MiniMax H3
# (minimax_h3 GGUF arch) and LTX-2.5 (gemma4 text encoder, ltxv fp32 tensors,
# tokenizer + embedding dequant fixes), plus the H3Lab and LTX25Lab control nodes.
# Idempotent: safe to rerun.
COMFY_PIN = "2f35f4a08176d993cded35dac3332be4f7287f41"   # ComfyUI 0.33.0
GGUF_PIN = "6ea2651e7df66d7585f6ffee804b20e92fb38b8a"    # city96 ComfyUI-GGUF
TURBO_COMMIT = "55fee864dd7b2976b1c4ce3c3d5f7968f181409f"

if "H3" not in globals() or "LTX" not in globals():
    raise RuntimeError("run cell 00 first")

import os, re, sys, time

os.environ.setdefault("HF_HUB_DISABLE_IMPLICIT_TOKEN", "1")


def git_sync(url, dest, ref):
    if not (dest / ".git").is_dir():
        if dest.exists():
            raise RuntimeError("path exists and is not a git repo: %s" % dest)
        rc, out = sh(["git", "clone", url, str(dest)], timeout=1800)
        if rc:
            raise RuntimeError("clone failed for %s:\n%s" % (url, out[-600:]))
    rc, _ = sh(["git", "-C", str(dest), "cat-file", "-e", ref + "^{commit}"])
    if rc:
        rc, out = sh(["git", "-C", str(dest), "fetch", "origin", ref, "--depth", "1"],
                     timeout=1800)
        if rc:
            rc, out = sh(["git", "-C", str(dest), "fetch", "origin"], timeout=1800)
            if rc:
                raise RuntimeError("fetch failed for %s:\n%s" % (dest, out[-600:]))
    rc, out = sh(["git", "-C", str(dest), "checkout", "-f", ref])
    if rc:
        raise RuntimeError("checkout failed for %s (%s):\n%s" % (dest, ref, out[-600:]))
    return sh(["git", "-C", str(dest), "rev-parse", "HEAD"])[1].strip()


def patch_loader():
    """Apply the H3 + LTX-2.5 fixes to city96's loader.py.

    1. accept the MiniMax H3 GGUF image architecture ("minimax_h3");
    2. accept the LTX-2.5 text-encoder architecture ("gemma4");
    3. force-dequantize three small ltxv tensors that must stay fp32
       (learnable registers + keyframe positional embedding);
    4. normalize the embedded gemma4 tokenizer_json blob to uint8 (the GGUF
       stores it as int8; ComfyUI's tokenizer rejects negative byte values).
    """
    p = GGUF_DIR / "loader.py"
    src = p.read_text(encoding="utf-8")
    notes = []

    h3_old = ('IMG_ARCH_LIST = {"flux", "sd1", "sdxl", "sd3", "aura", "hidream", '
              '"cosmos", "ltxv", "hyvid", "wan", "lumina2", "qwen_image"}')
    h3_new = ('IMG_ARCH_LIST = {"flux", "sd1", "sdxl", "sd3", "aura", "hidream", '
              '"cosmos", "ltxv", "hyvid", "wan", "lumina2", "qwen_image", '
              '"minimax_h3"}')
    if '"minimax_h3"' in src:
        notes.append("h3 arch already patched")
    elif src.count(h3_old) == 1:
        src = src.replace(h3_old, h3_new, 1)
        notes.append("h3 arch patched")
    else:
        raise RuntimeError("IMG_ARCH_LIST anchor found %d times (expected 1)"
                           % src.count(h3_old))

    ltx_old = ('TXT_ARCH_LIST = {"t5", "t5encoder", "llama", "qwen2vl", "qwen3", '
               '"qwen3vl", "gemma3"}')
    ltx_new = ('TXT_ARCH_LIST = {"t5", "t5encoder", "llama", "qwen2vl", "qwen3", '
               '"qwen3vl", "gemma3", "gemma4"}')
    if ltx_new in src:
        notes.append("gemma4 arch already patched")
    elif src.count(ltx_old) == 1:
        src = src.replace(ltx_old, ltx_new, 1)
        notes.append("gemma4 arch patched")
    else:
        raise RuntimeError("TXT_ARCH_LIST anchor found %d times (expected 1)"
                           % src.count(ltx_old))

    if "learnable_registers" in src:
        notes.append("ltxv dequant already present")
    else:
        anchor = "    # extra info to return\n"
        if src.count(anchor) != 1:
            raise RuntimeError("gguf_sd_loader return anchor found %d times "
                               "(expected 1)" % src.count(anchor))
        block = (
            '    if arch_str == "ltxv":\n'
            '        for _lk in list(state_dict.keys()):\n'
            '            if _lk.endswith(("audio_embeddings_connector.learnable_registers",\n'
            '                             "video_embeddings_connector.learnable_registers",\n'
            '                             "keyframes_abs_pos_embedding")):\n'
            '                state_dict[_lk] = dequantize_tensor(state_dict[_lk],\n'
            '                                                    dtype=torch.float32)\n\n')
        src = src.replace(anchor, block + anchor, 1)
        notes.append("ltxv force-dequant added")

    if "gguf gemma4: tokenizer_json normalized" in src:
        notes.append("gemma4 tokenizer already patched")
    else:
        idx = src.rfind("    return sd")
        if idx == -1 or idx < src.find("def gguf_clip_loader"):
            raise RuntimeError("could not find gguf_clip_loader tail for gemma4 patch")
        tj_block = (
            '    if arch == "gemma4" and "tokenizer_json" in sd:\n'
            '        _tj = sd["tokenizer_json"]\n'
            '        if isinstance(_tj, GGMLTensor):\n'
            '            _tj = torch.Tensor(_tj)\n'
            '        sd["tokenizer_json"] = (_tj.to(torch.int32) % 256).to(torch.uint8)\n'
            '        logging.info("gguf gemma4: tokenizer_json normalized to uint8")\n')
        src = src[:idx] + tj_block + src[idx:]
        notes.append("gemma4 tokenizer uint8 patch added")

    p.write_text(src, encoding="utf-8")
    rc, out = sh([sys.executable, "-c",
                  "import py_compile; py_compile.compile(%r, doraise=True)" % str(p)])
    if rc:
        raise RuntimeError("patched loader.py failed to compile:\n" + out[-800:])
    return ", ".join(notes)


def patch_ops():
    """Dequantize quantized embedding weights into an fp16 buffer, not fp32.

    ComfyUI runs the gemma4 text encoder in fp32, so the 1.0e9-element Q4_K
    embedding table would dequantize to a 3.75 GiB fp32 spike and OOM a T4.
    The result is cast back to the requested output dtype afterwards.
    """
    p = GGUF_DIR / "ops.py"
    src = p.read_text(encoding="utf-8")
    marker = "LTX25Lab: quantized embed dequant buffer in fp16"
    if marker in src:
        return "embedding fp16 dequant already present"
    old = ("            output_dtype = out_dtype\n"
           "            if self.weight.dtype == torch.float16 or self.weight.dtype "
           "== torch.bfloat16:\n"
           "                out_dtype = None\n")
    new = old + (
        "            if out_dtype is None or out_dtype == torch.float32 "
        "or out_dtype == torch.float64:\n"
        "                out_dtype = torch.float16  # " + marker + "\n")
    if src.count(old) != 1:
        raise RuntimeError("Embedding.forward_ggml_cast_weights anchor found %d times "
                           "(expected 1)" % src.count(old))
    src = src.replace(old, new, 1)
    p.write_text(src, encoding="utf-8")
    rc, out = sh([sys.executable, "-c",
                  "import py_compile; py_compile.compile(%r, doraise=True)" % str(p)])
    if rc:
        raise RuntimeError("patched ops.py failed to compile:\n" + out[-800:])
    return "embedding dequant forced to fp16"


H3_PACK_SRC = r'''"""H3Lab control nodes: conditioning/latent persistence + optional patches.

Three-stage split support (encode -> sample -> decode), plus:
  - H3Fp32TextBranch: partial fp16 overflow fix (condition_proj input upcast)
  - H3SagePatch: attention override for SageAttention (adapted from KJNodes'
    PathchSageAttentionKJ, MIT, trimmed to the sm75-safe kernel)
"""
import json
import logging
from pathlib import Path

import torch

H3_TORCH_NESTED = "__h3_torch_nested__"
H3_COMFY_NESTED = "__h3_comfy_nested__"


def _is_comfy_nested(obj):
    """comfy.nested_tensor.NestedTensor, duck-typed."""
    return (not isinstance(obj, torch.Tensor)
            and bool(getattr(obj, "is_nested", False))
            and hasattr(obj, "unbind"))


def _restore(obj):
    if isinstance(obj, dict):
        keys = set(obj.keys())
        if keys == {H3_TORCH_NESTED}:
            return torch.nested.nested_tensor(list(obj[H3_TORCH_NESTED]))
        if keys == {H3_COMFY_NESTED}:
            from comfy.nested_tensor import NestedTensor
            return NestedTensor(list(obj[H3_COMFY_NESTED]))
        return {key: _restore(value) for key, value in obj.items()}
    if isinstance(obj, tuple):
        return tuple(_restore(value) for value in obj)
    if isinstance(obj, list):
        return [_restore(value) for value in obj]
    return obj


def _summarize(obj):
    if _is_comfy_nested(obj):
        return {"kind": "comfy_nested_tensor", "parts": [_summarize(p) for p in obj.unbind()]}
    if isinstance(obj, torch.Tensor):
        if bool(getattr(obj, "is_nested", False)):
            return {"kind": "nested_tensor", "parts": [_summarize(p) for p in obj.unbind()]}
        return {"kind": "tensor", "shape": list(obj.shape), "dtype": str(obj.dtype),
                "device": str(obj.device),
                "finite": bool(torch.isfinite(obj).all().item())
                if obj.is_floating_point() else None}
    if isinstance(obj, dict):
        return {key: _summarize(value) for key, value in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_summarize(value) for value in obj]
    return {"kind": type(obj).__name__, "repr": repr(obj)[:200]}


def _detach_to_cpu(obj):
    if isinstance(obj, torch.Tensor):
        if bool(getattr(obj, "is_nested", False)):
            return {H3_TORCH_NESTED: [p.detach().cpu() for p in obj.unbind()]}
        return obj.detach().cpu()
    if _is_comfy_nested(obj):
        return {H3_COMFY_NESTED: [p.detach().cpu() for p in obj.unbind()]}
    if isinstance(obj, dict):
        return {key: _detach_to_cpu(value) for key, value in obj.items()}
    if isinstance(obj, tuple):
        return tuple(_detach_to_cpu(value) for value in obj)
    if isinstance(obj, list):
        return [_detach_to_cpu(value) for value in obj]
    return obj


class H3SaveConditioning:
    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "conditioning": ("CONDITIONING",), "samples": ("LATENT",),
            "store_dir": ("STRING", {"default": ""}),
            "key": ("STRING", {"default": "conditioning"})}}
    RETURN_TYPES = ()
    FUNCTION = "save"
    CATEGORY = "H3Lab"
    OUTPUT_NODE = True

    def save(self, conditioning, samples, store_dir, key):
        directory = Path(store_dir)
        directory.mkdir(parents=True, exist_ok=True)
        payload = {"schema_version": 1,
                   "conditioning": _detach_to_cpu(conditioning),
                   "samples": _detach_to_cpu(samples)}
        destination = directory / (key + ".pt")
        staged = destination.with_suffix(".pt.tmp")
        torch.save(payload, staged)
        staged.replace(destination)
        (directory / (key + ".summary.json")).write_text(json.dumps(
            {"key": key, "bytes": destination.stat().st_size,
             "conditioning": _summarize(payload["conditioning"]),
             "samples": _summarize(payload["samples"])}, indent=2), encoding="utf-8")
        logging.info("H3_SAVE_CONDITIONING %s", json.dumps(
            {"key": key, "bytes": destination.stat().st_size}, separators=(",", ":")))
        return {}


class H3LoadConditioning:
    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "store_dir": ("STRING", {"default": ""}),
            "key": ("STRING", {"default": "conditioning"})}}
    RETURN_TYPES = ("CONDITIONING", "LATENT")
    RETURN_NAMES = ("conditioning", "samples")
    FUNCTION = "load"
    CATEGORY = "H3Lab"

    def load(self, store_dir, key):
        source = Path(store_dir) / (key + ".pt")
        if not source.is_file():
            raise RuntimeError("Precomputed conditioning is missing: %s" % source)
        payload = torch.load(source, map_location="cpu", weights_only=False)
        if payload.get("schema_version") != 1:
            raise RuntimeError("Unsupported schema: %s" % payload.get("schema_version"))
        logging.info("H3_LOAD_CONDITIONING %s", json.dumps(
            {"key": key, "bytes": source.stat().st_size}, separators=(",", ":")))
        return (_restore(payload["conditioning"]), _restore(payload["samples"]))


class H3SaveLatent:
    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "samples": ("LATENT",),
            "store_dir": ("STRING", {"default": ""}),
            "key": ("STRING", {"default": "latent"})}}
    RETURN_TYPES = ("LATENT",)
    RETURN_NAMES = ("samples",)
    FUNCTION = "save"
    CATEGORY = "H3Lab"
    OUTPUT_NODE = True

    def save(self, samples, store_dir, key):
        directory = Path(store_dir)
        directory.mkdir(parents=True, exist_ok=True)
        path = directory / (key + ".pt")
        torch.save({"schema_version": 1, "key": key,
                    "samples": _detach_to_cpu(samples)}, path)
        logging.info("H3_SAVE_LATENT %s", json.dumps(
            {"key": key, "path": str(path), "bytes": path.stat().st_size,
             "summary": _summarize(samples)}, separators=(",", ":")))
        return (samples,)


class H3LoadLatent:
    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "store_dir": ("STRING", {"default": ""}),
            "key": ("STRING", {"default": "latent"})}}
    RETURN_TYPES = ("LATENT",)
    RETURN_NAMES = ("samples",)
    FUNCTION = "load"
    CATEGORY = "H3Lab"

    def load(self, store_dir, key):
        path = Path(store_dir) / (key + ".pt")
        if not path.is_file():
            raise RuntimeError("No latent at %s. Run the sampling cell first." % path)
        payload = torch.load(path, map_location="cpu", weights_only=False)
        if payload.get("schema_version") != 1:
            raise RuntimeError("Unsupported schema: %s" % payload.get("schema_version"))
        samples = _restore(payload["samples"])
        logging.info("H3_LOAD_LATENT %s", json.dumps(
            {"key": key, "bytes": path.stat().st_size,
             "summary": _summarize(samples)}, separators=(",", ":")))
        return (samples,)


class H3LatentProbe:
    """Report the sampled latent. Raises on non-finite values, after saving the report."""

    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "samples": ("LATENT",),
            "filename_prefix": ("STRING", {"default": "h3_lab/probe"})}}
    RETURN_TYPES = ("LATENT",)
    RETURN_NAMES = ("samples",)
    FUNCTION = "save_summary"
    CATEGORY = "H3Lab"
    OUTPUT_NODE = True

    def save_summary(self, samples, filename_prefix):
        import folder_paths
        summary = _summarize(samples)
        packed = samples.get("samples")
        parts = packed.unbind() if (_is_comfy_nested(packed)
                                    or getattr(packed, "is_nested", False)) else [packed]
        finite = [bool(torch.isfinite(p).all().item()) for p in parts
                  if isinstance(p, torch.Tensor)]
        output_path = (Path(folder_paths.get_output_directory())
                       / (filename_prefix + ".latent_summary.json"))
        output_path.parent.mkdir(parents=True, exist_ok=True)
        payload = {"summary": summary,
                   "shapes": [list(p.shape) for p in parts if isinstance(p, torch.Tensor)],
                   "finite": finite}
        output_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
        logging.info("H3_LATENT_PROBE %s", json.dumps(payload["shapes"], separators=(",", ":")))
        if not finite or not all(finite):
            raise RuntimeError("Sampled latent contains non-finite values: %s" % summary)
        return (samples,)


class H3Fp32TextBranch:
    """Run the text branch in fp32 (condition_proj input absmax ~15424 in fp16)."""

    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {"model": ("MODEL",)}}
    RETURN_TYPES = ("MODEL",)
    RETURN_NAMES = ("model",)
    FUNCTION = "apply"
    CATEGORY = "H3Lab"

    def apply(self, model):
        target = getattr(getattr(model, "model", None), "diffusion_model", None)
        proj = getattr(target, "condition_proj", None)
        if proj is None:
            logging.warning("H3_FP32_TEXT_BRANCH no condition_proj found; nothing patched")
            return (model,)
        if getattr(proj, "_h3_fp32_text_branch", False):
            logging.info("H3_FP32_TEXT_BRANCH already_applied")
            return (model,)
        original = proj.forward

        def forward(x, *args, **kwargs):
            if isinstance(x, torch.Tensor) and x.dtype == torch.float16:
                x = x.to(torch.float32)
            return original(x, *args, **kwargs)

        proj.forward = forward
        proj._h3_fp32_text_branch = True
        logging.info("H3_FP32_TEXT_BRANCH applied to condition_proj")
        return (model,)


class H3SagePatch:
    """Route model attention through SageAttention (int8 QK, fp16 PV)."""

    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "model": ("MODEL",),
            "mode": (["sageattn_qk_int8_pv_fp16_cuda", "auto"],)}}
    RETURN_TYPES = ("MODEL",)
    RETURN_NAMES = ("model",)
    FUNCTION = "patch"
    CATEGORY = "H3Lab"

    def patch(self, model, mode):
        import torch
        from sageattention import sageattn, sageattn_qk_int8_pv_fp16_cuda
        from comfy.ldm.modules.attention import attention_pytorch
        try:
            from comfy.ldm.modules.attention import wrap_attn
        except Exception:
            def wrap_attn(f):
                f.__wrapped__ = f
                return f

        if mode == "auto":
            def kernel(q, k, v, is_causal=False, attn_mask=None, tensor_layout="NHD"):
                return sageattn(q, k, v, is_causal=is_causal, attn_mask=attn_mask,
                                tensor_layout=tensor_layout)
        else:
            def kernel(q, k, v, is_causal=False, attn_mask=None, tensor_layout="NHD"):
                return sageattn_qk_int8_pv_fp16_cuda(
                    q, k, v, is_causal=is_causal, attn_mask=attn_mask,
                    pv_accum_dtype="fp32", tensor_layout=tensor_layout)

        kernel = torch.compiler.disable()(kernel)

        @wrap_attn
        def attention_sage(q, k, v, heads, mask=None, attn_precision=None,
                           skip_reshape=False, skip_output_reshape=False, **kwargs):
            if kwargs.get("low_precision_attention", True) is False:
                return attention_pytorch(q, k, v, heads, mask=mask,
                                         skip_reshape=skip_reshape,
                                         skip_output_reshape=skip_output_reshape, **kwargs)
            in_dtype = v.dtype
            if (q.dtype == torch.float32 or k.dtype == torch.float32
                    or v.dtype == torch.float32):
                q, k, v = q.to(torch.float16), k.to(torch.float16), v.to(torch.float16)
            if skip_reshape:
                b, _, _, dim_head = q.shape
                tensor_layout = "HND"
            else:
                b, _, dim_head = q.shape
                dim_head //= heads
                q, k, v = map(lambda t: t.view(b, -1, heads, dim_head), (q, k, v))
                tensor_layout = "NHD"
            if mask is not None:
                if mask.ndim == 2:
                    mask = mask.unsqueeze(0)
                if mask.ndim == 3:
                    mask = mask.unsqueeze(1)
            seq_dim = 2 if tensor_layout == "HND" else 1
            if any((t.shape[seq_dim] - 1) * t.stride(seq_dim) >= 2 ** 31 for t in (q, k, v)):
                q, k, v = q.contiguous(), k.contiguous(), v.contiguous()
            out = kernel(q, k, v, attn_mask=mask, is_causal=False,
                         tensor_layout=tensor_layout).to(in_dtype)
            if tensor_layout == "HND":
                if not skip_output_reshape:
                    out = out.transpose(1, 2).reshape(b, -1, heads * dim_head)
            else:
                if skip_output_reshape:
                    out = out.transpose(1, 2)
                else:
                    out = out.reshape(b, -1, heads * dim_head)
            return out

        clone = model.clone()
        clone.model_options["transformer_options"]["optimized_attention_override"] = (
            lambda func, *args, **kwargs: attention_sage.__wrapped__(*args, **kwargs))
        logging.info("[H3Lab] sage attention override active (mode=%s)", mode)
        return (clone,)


NODE_CLASS_MAPPINGS = {
    "H3SaveConditioning": H3SaveConditioning,
    "H3LoadConditioning": H3LoadConditioning,
    "H3SaveLatent": H3SaveLatent,
    "H3LoadLatent": H3LoadLatent,
    "H3LatentProbe": H3LatentProbe,
    "H3Fp32TextBranch": H3Fp32TextBranch,
    "H3SagePatch": H3SagePatch,
}

NODE_DISPLAY_NAME_MAPPINGS = {
    "H3SaveConditioning": "H3 Save Conditioning",
    "H3LoadConditioning": "H3 Load Conditioning",
    "H3SaveLatent": "H3 Save Latent",
    "H3LoadLatent": "H3 Load Latent",
    "H3LatentProbe": "H3 Latent Probe",
    "H3Fp32TextBranch": "H3 Fp32 Text Branch",
    "H3SagePatch": "H3 Sage Attention Patch",
}
'''

LTX_PACK_SRC = '''"""LTX25Lab control nodes: conditioning/latent persistence across split jobs."""
import json
import logging
from pathlib import Path

import torch

LTX_TORCH_NESTED = "__ltx_torch_nested__"
LTX_COMFY_NESTED = "__ltx_comfy_nested__"


def _is_comfy_nested(obj):
    return (not isinstance(obj, torch.Tensor)
            and bool(getattr(obj, "is_nested", False))
            and hasattr(obj, "unbind"))


def _restore(obj):
    if isinstance(obj, dict):
        keys = set(obj.keys())
        if keys == {LTX_TORCH_NESTED}:
            return torch.nested.nested_tensor(list(obj[LTX_TORCH_NESTED]))
        if keys == {LTX_COMFY_NESTED}:
            from comfy.nested_tensor import NestedTensor
            return NestedTensor(list(obj[LTX_COMFY_NESTED]))
        return {key: _restore(value) for key, value in obj.items()}
    if isinstance(obj, tuple):
        return tuple(_restore(value) for value in obj)
    if isinstance(obj, list):
        return [_restore(value) for value in obj]
    return obj


def _summarize(obj):
    if _is_comfy_nested(obj):
        return {"kind": "comfy_nested_tensor", "parts": [_summarize(p) for p in obj.unbind()]}
    if isinstance(obj, torch.Tensor):
        if bool(getattr(obj, "is_nested", False)):
            return {"kind": "nested_tensor", "parts": [_summarize(p) for p in obj.unbind()]}
        return {"kind": "tensor", "shape": list(obj.shape), "dtype": str(obj.dtype),
                "device": str(obj.device),
                "finite": bool(torch.isfinite(obj).all().item())
                if obj.is_floating_point() else None}
    if isinstance(obj, dict):
        return {key: _summarize(value) for key, value in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_summarize(value) for value in obj]
    return {"kind": type(obj).__name__, "repr": repr(obj)[:200]}


def _detach_to_cpu(obj):
    if isinstance(obj, torch.Tensor):
        if bool(getattr(obj, "is_nested", False)):
            return {LTX_TORCH_NESTED: [p.detach().cpu() for p in obj.unbind()]}
        return obj.detach().cpu()
    if _is_comfy_nested(obj):
        return {LTX_COMFY_NESTED: [p.detach().cpu() for p in obj.unbind()]}
    if isinstance(obj, dict):
        return {key: _detach_to_cpu(value) for key, value in obj.items()}
    if isinstance(obj, tuple):
        return tuple(_detach_to_cpu(value) for value in obj)
    if isinstance(obj, list):
        return [_detach_to_cpu(value) for value in obj]
    return obj


class LTX25SaveConditioning:
    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "conditioning": ("CONDITIONING",),
            "store_dir": ("STRING", {"default": ""}),
            "key": ("STRING", {"default": "conditioning"})}}
    RETURN_TYPES = ()
    FUNCTION = "save"
    CATEGORY = "LTX25Lab"
    OUTPUT_NODE = True

    def save(self, conditioning, store_dir, key):
        directory = Path(store_dir)
        directory.mkdir(parents=True, exist_ok=True)
        payload = {"schema_version": 1, "key": key,
                   "conditioning": _detach_to_cpu(conditioning)}
        destination = directory / (key + ".pt")
        staged = destination.with_suffix(".pt.tmp")
        torch.save(payload, staged)
        staged.replace(destination)
        (directory / (key + ".summary.json")).write_text(json.dumps(
            {"key": key, "bytes": destination.stat().st_size,
             "conditioning": _summarize(payload["conditioning"])}, indent=2),
            encoding="utf-8")
        logging.info("LTX25_SAVE_CONDITIONING %s", json.dumps(
            {"key": key, "bytes": destination.stat().st_size}, separators=(",", ":")))
        return {}


class LTX25LoadConditioning:
    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "store_dir": ("STRING", {"default": ""}),
            "key": ("STRING", {"default": "conditioning"})}}
    RETURN_TYPES = ("CONDITIONING",)
    RETURN_NAMES = ("conditioning",)
    FUNCTION = "load"
    CATEGORY = "LTX25Lab"

    def load(self, store_dir, key):
        source = Path(store_dir) / (key + ".pt")
        if not source.is_file():
            raise RuntimeError("Precomputed conditioning is missing: %s" % source)
        payload = torch.load(source, map_location="cpu", weights_only=False)
        if payload.get("schema_version") != 1:
            raise RuntimeError("Unsupported schema: %s" % payload.get("schema_version"))
        logging.info("LTX25_LOAD_CONDITIONING %s", json.dumps(
            {"key": key, "bytes": source.stat().st_size}, separators=(",", ":")))
        return (_restore(payload["conditioning"]),)


class LTX25SaveLatent:
    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "samples": ("LATENT",),
            "store_dir": ("STRING", {"default": ""}),
            "key": ("STRING", {"default": "latent"})}}
    RETURN_TYPES = ("LATENT",)
    RETURN_NAMES = ("samples",)
    FUNCTION = "save"
    CATEGORY = "LTX25Lab"
    OUTPUT_NODE = True

    def save(self, samples, store_dir, key):
        directory = Path(store_dir)
        directory.mkdir(parents=True, exist_ok=True)
        path = directory / (key + ".pt")
        staged = path.with_suffix(".pt.tmp")
        torch.save({"schema_version": 1, "key": key,
                    "samples": _detach_to_cpu(samples)}, staged)
        staged.replace(path)
        (directory / (key + ".summary.json")).write_text(json.dumps(
            {"key": key, "bytes": path.stat().st_size,
             "samples": _summarize(samples)}, indent=2), encoding="utf-8")
        logging.info("LTX25_SAVE_LATENT %s", json.dumps(
            {"key": key, "path": str(path), "bytes": path.stat().st_size,
             "summary": _summarize(samples)}, separators=(",", ":")))
        return (samples,)


class LTX25LoadLatent:
    @classmethod
    def INPUT_TYPES(cls):
        return {"required": {
            "store_dir": ("STRING", {"default": ""}),
            "key": ("STRING", {"default": "latent"})}}
    RETURN_TYPES = ("LATENT",)
    RETURN_NAMES = ("samples",)
    FUNCTION = "load"
    CATEGORY = "LTX25Lab"

    def load(self, store_dir, key):
        path = Path(store_dir) / (key + ".pt")
        if not path.is_file():
            raise RuntimeError("No latent at %s. Run the sampling stage first." % path)
        payload = torch.load(path, map_location="cpu", weights_only=False)
        if payload.get("schema_version") != 1:
            raise RuntimeError("Unsupported schema: %s" % payload.get("schema_version"))
        logging.info("LTX25_LOAD_LATENT %s", json.dumps(
            {"key": key, "bytes": path.stat().st_size,
             "summary": _summarize(payload["samples"])}, separators=(",", ":")))
        return (_restore(payload["samples"]),)


NODE_CLASS_MAPPINGS = {
    "LTX25SaveConditioning": LTX25SaveConditioning,
    "LTX25LoadConditioning": LTX25LoadConditioning,
    "LTX25SaveLatent": LTX25SaveLatent,
    "LTX25LoadLatent": LTX25LoadLatent,
}

NODE_DISPLAY_NAME_MAPPINGS = {
    "LTX25SaveConditioning": "LTX25 Save Conditioning",
    "LTX25LoadConditioning": "LTX25 Load Conditioning",
    "LTX25SaveLatent": "LTX25 Save Latent",
    "LTX25LoadLatent": "LTX25 Load Latent",
}
'''

rows, errors = [], []
live = LiveBox("01", "01 . base stack - running", kind="run")
stage = {"label": "updating repositories"}
_started_01 = time.perf_counter()


def _tick01(live):
    live.set(rows=list(rows) + [("elapsed", fmt_secs(time.perf_counter() - _started_01))],
             note=stage["label"] + " ...")


ticker = Ticker(live, _tick01, 2.0).start()

try:
    stage["label"] = "updating ComfyUI repository"
    head = git_sync("https://github.com/Comfy-Org/ComfyUI.git", COMFY_DIR, COMFY_PIN)
    if head != COMFY_PIN:
        raise RuntimeError("ComfyUI HEAD is %s but %s was requested; refusing to run a "
                           "different version silently" % (head[:12], COMFY_PIN[:12]))
    rows.append(("ComfyUI", head[:12] + " (pinned 0.33.0)"))
except Exception as exc:
    errors.append("ComfyUI: %s" % exc)

try:
    stage["label"] = "installing ComfyUI requirements (pip)"
    req = COMFY_DIR / "requirements.txt"
    keep, count = [], 0
    for line in req.read_text(encoding="utf-8").splitlines():
        stripped = line.strip()
        if stripped and not stripped.startswith("#"):
            name = re.split(r"[<>=!~ ]", stripped, maxsplit=1)[0].strip().lower()
            if name in ("torch", "torchvision", "torchaudio"):
                continue
            count += 1
        keep.append(line)
    filtered = WORK / "comfy_requirements.txt"
    filtered.write_text("\n".join(keep) + "\n", encoding="utf-8")
    rc, out = sh([sys.executable, "-m", "pip", "install", "-q", "-r", str(filtered)],
                 timeout=2400)
    if rc:
        raise RuntimeError("pip install -r requirements failed:\n" + out[-800:])
    rows.append(("python deps", "%d packages from requirements.txt (torch kept as "
                 "Colab's)" % count))
except Exception as exc:
    errors.append("ComfyUI deps: %s" % exc)

try:
    stage["label"] = "updating ComfyUI-GGUF + H3/LTX patches"
    head = git_sync("https://github.com/city96/ComfyUI-GGUF.git", GGUF_DIR, GGUF_PIN)
    req2 = GGUF_DIR / "requirements.txt"
    if req2.is_file():
        rc, out = sh([sys.executable, "-m", "pip", "install", "-q", "-r", str(req2)],
                     timeout=1200)
        if rc:
            raise RuntimeError("GGUF node requirements failed:\n" + out[-500:])
    notes = patch_loader()
    notes += "; " + patch_ops()
    rows.append(("ComfyUI-GGUF", head[:12] + " (pinned)"))
    rows.append(("GGUF patches", notes))
except Exception as exc:
    errors.append("ComfyUI-GGUF: %s" % exc)

try:
    stage["label"] = "updating H3 Turbo nodes"
    head = git_sync("https://github.com/Larryvrh/ComfyUI-MiniMax-H3-Turbo.git",
                    TURBO_DIR, TURBO_COMMIT)
    grid = TURBO_DIR / "h3_silu_temb_grid.safetensors"
    if not grid.is_file():
        raise RuntimeError("h3_silu_temb_grid.safetensors missing from the Turbo "
                           "checkout; the LoRA would apply only partially and silently")
    rows.append(("H3 Turbo nodes", head[:12] + ", silu grid %d bytes"
                 % grid.stat().st_size))
except Exception as exc:
    errors.append("Turbo nodes: %s" % exc)

try:
    stage["label"] = "installing custom node requirements"
    node_deps = "no requirements.txt found"
    dep_rows = []
    for label, d in (("GGUF", GGUF_DIR), ("Turbo", TURBO_DIR),
                     ("Spectrum", SPECTRUM_DIR), ("FP16Safe", FP16SAFE_DIR)):
        req = d / "requirements.txt"
        if d.is_dir() and req.is_file():
            rc, out = sh([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)],
                         timeout=1200)
            if rc:
                raise RuntimeError("%s requirements failed:\n%s" % (label, out[-500:]))
            dep_rows.append(label)
    if dep_rows:
        node_deps = ", ".join(dep_rows)
    rows.append(("node deps", node_deps))
except Exception as exc:
    errors.append("custom node deps: %s" % exc)

try:
    stage["label"] = "writing H3Lab + LTX25Lab control nodes"
    H3_CONTROL_DIR.mkdir(parents=True, exist_ok=True)
    (H3_CONTROL_DIR / "__init__.py").write_text(H3_PACK_SRC, encoding="utf-8")
    LTX_CONTROL_DIR.mkdir(parents=True, exist_ok=True)
    (LTX_CONTROL_DIR / "__init__.py").write_text(LTX_PACK_SRC, encoding="utf-8")
    for pack, label in ((H3_CONTROL_DIR, "H3Lab"), (LTX_CONTROL_DIR, "LTX25Lab")):
        rc, out = sh([sys.executable, "-c",
                      "import py_compile; py_compile.compile(%r, doraise=True)"
                      % str(pack / "__init__.py")])
        if rc:
            raise RuntimeError("%s control pack failed to compile:\n%s"
                               % (label, out[-600:]))
    rows.append(("control nodes", "H3Lab (7 nodes) + LTX25Lab (4 nodes) compiled"))
except Exception as exc:
    errors.append("control nodes: %s" % exc)

try:
    stage["label"] = "checking video encoder (av)"
    rc, out = sh([sys.executable, "-c", "import av; print(av.__version__)"])
    if rc:
        rc, out = sh([sys.executable, "-m", "pip", "install", "-q", "av"], timeout=900)
        if rc:
            raise RuntimeError("av install failed:\n" + out[-500:])
        rc, out = sh([sys.executable, "-c", "import av; print(av.__version__)"])
    rows.append(("pyav", out.strip()))
except Exception as exc:
    errors.append("pyav: %s" % exc)

ticker.stop_now()
if errors:
    live.finish("01 . base stack - incomplete", rows=rows, kind="err",
                note="<br>".join(errors) + "<br>Fix the issue and rerun this cell; it is "
                     "safe to run repeatedly.")
    raise RuntimeError("; ".join(errors))
live.finish("01 . base stack ready", rows=rows, kind="ok",
            note="One pinned ComfyUI 0.33.0 + GGUF loader with the H3 (minimax_h3) and "
                 "LTX-2.5 (gemma4 / ltxv / embedding) patches. The Setup tab installs "
                 "H3 method nodes on demand. Next: 02 downloads both model sets, 03 "
                 "launches the Studio.")


In [ ]:
# @title 02 . models { display-mode: "form" }
# Downloads the MiniMax H3 (Q3/Q4 GGUF + FL2VA/Ref2VA + VAEs + turbo LoRA) and
# LTX-2.5 (22B distilled GGUF + gemma4 text encoder + VAEs + x2 upscaler) stacks.
# Resumable and skippable: files already present at the right size are left alone.
MODEL_SET = "both" #@param ["both", "h3", "ltx"]
H3_DIT_CHOICE = "abiray_Q3_K_M" #@param ["abiray_Q3_K_M", "abiray_Q4_K_M"]
INCLUDE_TEXT_ENCODER = "auto" #@param ["auto", "yes", "no"]
TURBO_LORAS = "v4 only" #@param ["v4 only", "v4 + v1", "none"]
REF2VA_CHOICE = "Q3_K_M" #@param ["Q3_K_M", "Q4_K_M", "none"]
FORCE_REDOWNLOAD = False #@param {type:"boolean"}

if "H3" not in globals() or "LTX" not in globals():
    raise RuntimeError("run cell 00 first")
if not (COMFY_DIR / ".git").is_dir():
    raise RuntimeError("ComfyUI is not installed yet; run cell 01 first")

import os, re, time
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_IMPLICIT_TOKEN", "1")

try:
    from huggingface_hub import hf_hub_download
except ImportError:
    rc, out = sh([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
    from huggingface_hub import hf_hub_download

try:
    from huggingface_hub.utils import disable_progress_bars
    disable_progress_bars()
except Exception:
    pass

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass

WANT_H3 = MODEL_SET in ("both", "h3")
WANT_LTX = MODEL_SET in ("both", "ltx")

# ---------------------------------------------------------------- H3 catalog --
MODEL_REPO = "Comfy-Org/MiniMax-H3"
MODEL_REVISION = "0543966fbdce5ba05709a8f2031c94bdba629b4a"
DIT_CHOICES = {
    "abiray_Q3_K_M": {
        "repo": "Abiray/MiniMax-H3-Pruned-GGUF",
        "file": "MiniMax-H3-FL2VA-Pruned-Q3_K_M.gguf",
        "bytes": 8902845536,
        "sha256": "c844d70dbb4df67b7d9605c669bff35d9c631bd0507143cf0276f984de32247d"},
    "abiray_Q4_K_M": {
        "repo": "Abiray/MiniMax-H3-Pruned-GGUF",
        "file": "MiniMax-H3-FL2VA-Pruned-Q4_K_M.gguf",
        "bytes": 11564180576,
        "sha256": "d74e644906fe95b8c7bf4837ba9d3f9392925f7301675a0993d419b62a4824a3"},
}
REF2VA_CHOICES = {
    "Q3_K_M": {
        "repo": "Abiray/MiniMax-H3-Pruned-GGUF",
        "file": "MiniMax-H3-Ref2VA-Pruned-Q3_K_M.gguf",
        "bytes": 8902845536,
        "sha256": "6e22b83e7c2213018ab0e66542633564fc6ffc05df25ed6fce2285716f335e58"},
    "Q4_K_M": {
        "repo": "Abiray/MiniMax-H3-Pruned-GGUF",
        "file": "MiniMax-H3-Ref2VA-Pruned-Q4_K_M.gguf",
        "bytes": 11564180576,
        "sha256": "a67ff28c56f0bcf05f7a5edb2d215f2dce5c302587d4caf748e2e61e1ffea736"},
}
TURBO_LORA_CHOICES = {
    "v4_step600_ema": {
        "repo": "larryvrh/MiniMax-H3-Turbo-Lora",
        "file": "minimax_h3_turbo_v4_step600_ema.safetensors",
        "bytes": 779849816,
        "sha256": "5f3a626cd72c93a8b9318d6760c510bc5092d2ab13aaba1f932c5bab07a416d3"},
    "v1_850_ema": {
        "repo": "larryvrh/MiniMax-H3-Turbo-Lora",
        "file": "minimax_h3_turbo_4step_ema_ckpt850.safetensors",
        "bytes": 779849816,
        "sha256": "5a6eeba171cf183020a4ad48774bb2968f29f8168afd6ec17a04987f3528b4ea"},
}
TE_FILE = {"path": "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
           "name": "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
           "bytes": 15687142551,
           "sha256": "35a88d51044231fe332301d7a62aa81e3f2cba62febeb446e2c1e3e0ef76f2c6"}
VAE_FILES = {
    "video": {"path": "vae/minimax_h3_video_vae_fp16.safetensors",
              "name": "minimax_h3_video_vae_fp16.safetensors",
              "bytes": 5207808496,
              "sha256": "7c1f131492e7eddacaac9069a61b81bdd39de5cc96561e677c5eab1cdce5e522"},
    "audio": {"path": "vae/minimax_h3_audio_vae_fp32.safetensors",
              "name": "minimax_h3_audio_vae_fp32.safetensors",
              "bytes": 605254808,
              "sha256": "8e505d95dd1561d47abd43d4238fd40d9bb1ae9e147ed0a4cba778d76ae4db48"},
}

# --------------------------------------------------------------- LTX catalog --
DIT_REPO, DIT_NEEDLE = "unsloth/LTX-2.5-GGUF", "UD-Q2_K_XL.gguf"
TE_REPO, TE_NEEDLE = "ChrisColeTech/LTX-2.5-uncensored-v1.1-FP8", \
    "gemma4_12b_ltx25_uncensored-Q4_K_M.gguf"
CC_REPO = "ChrisColeTech/LTX-2.5-uncensored-v1.1-FP8"
UPSCALER_REPO, UPSCALER_NEEDLE = "ChrisColeTech/LTX-2.5-uncensored-v1.1-FP8", \
    "ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors"
VAE_BUCKET = "https://huggingface.co/buckets/A20labs/LTX-2.5-bucket/resolve/vae/%s"

LTX_SPECS = [
    {"label": "LTX DiT  22B distilled",
     "dest": MODELS / "unet" / "ltx-2.5-22b-distilled-UD-Q2_K_XL.gguf",
     "refs": [("repo", DIT_REPO, DIT_NEEDLE)],
     "bytes": 9500270976,
     "sha256": "d6fdc7e117f1a9780bdbe15c898d63c12a7d747847f54b276b69a6be5ce3c57d"},
    {"label": "LTX TE  gemma4-12B Q4_K_M",
     "dest": MODELS / "text_encoders" / "gemma4_12b_ltx25_uncensored-Q4_K_M.gguf",
     "refs": [("repo", TE_REPO, TE_NEEDLE)],
     "bytes": 7509634752,
     "sha256": "79f1a1e0744e74713ed192ab3bca224aa5ce9cd20b08e52e1ad08fd32778e40b"},
    {"label": "LTX video VAE",
     "dest": MODELS / "vae" / "ltx25_video_vae.safetensors",
     "refs": [("url", VAE_BUCKET % "ltx-2.5-video-vae-conv-bf16.safetensors"),
              ("repo", CC_REPO, "video_vae")],
     "bytes": 1452269922,
     "sha256": "685b06ee3d9b2039647698fc4ea33175112462fc374e2777312c907897dfce8d"},
    {"label": "LTX audio VAE",
     "dest": MODELS / "vae" / "ltx25_audio_vae.safetensors",
     "refs": [("url", VAE_BUCKET % "ltx-2.5-audio-vae-bf16.safetensors"),
              ("repo", CC_REPO, "audio_vae")],
     "bytes": 364866540,
     "sha256": "c52733d37f6a7fb7949c3dc0fb468c6cb2169e4d836983a73babb9f0d54837a5"},
    {"label": "LTX x2 latent upscaler",
     "dest": MODELS / "latent_upscale_models" /
             "ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors",
     "refs": [("repo", UPSCALER_REPO, UPSCALER_NEEDLE),
              ("repo", "unsloth/LTX-2.5-GGUF", UPSCALER_NEEDLE)],
     "bytes": 995778752,
     "sha256": "eb5a71fe4068ee87ccdb1c3aa635e547ca76bd2d30ae20ae889f2c325c0677e8"},
]
LTX_FILES = {
    "dit": {"path": LTX_SPECS[0]["dest"].relative_to(MODELS).as_posix(),
            "name": LTX_SPECS[0]["dest"].name},
    "te": {"path": LTX_SPECS[1]["dest"].relative_to(MODELS).as_posix(),
           "name": LTX_SPECS[1]["dest"].name},
    "vae_video": {"path": LTX_SPECS[2]["dest"].relative_to(MODELS).as_posix(),
                  "name": LTX_SPECS[2]["dest"].name},
    "vae_audio": {"path": LTX_SPECS[3]["dest"].relative_to(MODELS).as_posix(),
                  "name": LTX_SPECS[3]["dest"].name},
    "upscaler": {"path": LTX_SPECS[4]["dest"].relative_to(MODELS).as_posix(),
                 "name": LTX_SPECS[4]["dest"].name},
}

for sub in ("diffusion_models", "unet", "loras", "vae", "text_encoders",
            "latent_upscale_models"):
    (MODELS / sub).mkdir(parents=True, exist_ok=True)

DEFAULT_REF2VA_CHOICE = REF2VA_CHOICE if REF2VA_CHOICE in REF2VA_CHOICES else "Q3_K_M"

if WANT_H3:
    if INCLUDE_TEXT_ENCODER == "yes":
        need_te, te_reason = True, "forced by config"
    elif INCLUDE_TEXT_ENCODER == "no":
        need_te, te_reason = False, "disabled by config"
    else:
        cached = [p for p in COND_DIR.glob("*.pt")
                  if p.is_file() and p.stat().st_size > 1024 * 1024] \
            if COND_DIR.is_dir() else []
        need_te = not cached
        te_reason = ("%d cached conditioning blob(s) found" % len(cached) if cached
                     else "no cached conditioning blob")
else:
    need_te, te_reason = False, "H3 not selected"

MODEL_SET_LABEL = {"both": "both (H3 + LTX-2.5)",
                   "h3": "h3 (MiniMax H3 only)",
                   "ltx": "ltx (LTX-2.5 only)"}.get(MODEL_SET, MODEL_SET)
rows, errors = [("model set", MODEL_SET_LABEL)], []
live = LiveBox("02", "02 . models - running", kind="run")
state = {"label": "checking local caches", "started": time.perf_counter(),
         "expected": 0}


def _incomplete_bytes():
    total = 0
    for p in MODELS.rglob("*.incomplete"):
        try:
            total += p.stat().st_size
        except Exception:
            pass
    for p in MODELS.rglob("*.part"):
        try:
            total += p.stat().st_size
        except Exception:
            pass
    return total


def _tick02(live):
    live.set(rows=list(rows),
             note="%s · %s of %s downloaded · elapsed %s" % (
                 state["label"], gb(_incomplete_bytes()), gb(state["expected"]),
                 fmt_secs(time.perf_counter() - state["started"])))


ticker = Ticker(live, _tick02, 2.0).start()


def ensure_hf(label, repo, filename, local_dir, expected_bytes, revision=None):
    dest = Path(local_dir) / filename
    if dest.is_file() and dest.stat().st_size == expected_bytes and not FORCE_REDOWNLOAD:
        rows.append((label, "%s  cached" % gb(expected_bytes)))
        return dest
    state["label"] = "downloading " + label
    state["started"] = time.perf_counter()
    state["expected"] = expected_bytes
    try:
        hf_hub_download(repo_id=repo, filename=filename, revision=revision,
                        local_dir=str(local_dir), token=HF_TOKEN)
    except Exception as exc:
        errors.append("%s: %s: %s" % (label, type(exc).__name__, str(exc)[:220]))
        rows.append((label, "FAILED - see note"))
        return None
    if not dest.is_file() or dest.stat().st_size != expected_bytes:
        got = dest.stat().st_size if dest.is_file() else 0
        errors.append("%s: size mismatch (got %s, expected %s)"
                      % (label, got, expected_bytes))
        rows.append((label, "SIZE MISMATCH"))
        return None
    rows.append((label, "%s  downloaded" % gb(expected_bytes)))
    return dest


if WANT_H3:
    dit = DIT_CHOICES[H3_DIT_CHOICE]
    ensure_hf("H3 DiT " + H3_DIT_CHOICE, dit["repo"], dit["file"],
              MODELS / "diffusion_models", dit["bytes"])

    WANTED_LORAS = {"v4 only": ["v4_step600_ema"],
                    "v4 + v1": ["v4_step600_ema", "v1_850_ema"],
                    "none": []}[TURBO_LORAS]
    for choice in WANTED_LORAS:
        spec = TURBO_LORA_CHOICES[choice]
        ensure_hf("H3 LoRA " + choice, spec["repo"], spec["file"],
                  MODELS / "loras", spec["bytes"])
    if not WANTED_LORAS:
        rows.append(("H3 turbo LoRAs", "skipped (none)"))

    ensure_hf("H3 video VAE", MODEL_REPO, VAE_FILES["video"]["path"], MODELS,
              VAE_FILES["video"]["bytes"], revision=MODEL_REVISION)
    ensure_hf("H3 audio VAE", MODEL_REPO, VAE_FILES["audio"]["path"], MODELS,
              VAE_FILES["audio"]["bytes"], revision=MODEL_REVISION)

    if need_te:
        ensure_hf("H3 text encoder", MODEL_REPO, TE_FILE["path"], MODELS,
                  TE_FILE["bytes"], revision=MODEL_REVISION)
    else:
        rows.append(("H3 text encoder", "skipped (%s)" % te_reason))

    if REF2VA_CHOICE != "none":
        ref = REF2VA_CHOICES[REF2VA_CHOICE]
        ensure_hf("H3 Ref2VA " + REF2VA_CHOICE, ref["repo"], ref["file"],
                  MODELS / "diffusion_models", ref["bytes"])
    else:
        rows.append(("H3 Ref2VA", "skipped (none)"))

    H3["models_ok"] = (
        (MODELS / "diffusion_models" / dit["file"]).is_file()
        and (MODELS / VAE_FILES["video"]["path"]).is_file()
        and (MODELS / VAE_FILES["audio"]["path"]).is_file()
        and all((MODELS / "loras" / TURBO_LORA_CHOICES[c]["file"]).is_file()
                for c in WANTED_LORAS)
        and (not need_te or (MODELS / TE_FILE["path"]).is_file()))
    H3["ref2va_ok"] = (REF2VA_CHOICE != "none"
                       and (MODELS / "diffusion_models"
                            / REF2VA_CHOICES[REF2VA_CHOICE]["file"]).is_file())
else:
    H3["models_ok"] = False
    H3["ref2va_ok"] = False
    rows.append(("H3 models", "skipped (MODEL_SET=%s)" % MODEL_SET))


def hf_find(repo, needle):
    r = requests.get("https://huggingface.co/api/models/" + repo, timeout=45)
    r.raise_for_status()
    files = [s["rfilename"] for s in r.json().get("siblings", [])]
    for name in files:
        if needle.lower() in name.lower():
            return "https://huggingface.co/%s/resolve/main/%s" % (repo, name)
    raise RuntimeError("no file matching %r in %s (first files: %s)"
                       % (needle, repo, files[:30]))


def curl_resume(url, dest, timeout_s=7200):
    part = Path(str(dest) + ".part")
    rc, out = sh(["bash", "-c",
                  "curl -L --fail --retry 4 --retry-delay 5 -C - --connect-timeout 30 "
                  "-o %s %s" % (str(part).replace("'", "'\\''"),
                                url.replace("'", "'\\''"))], timeout=timeout_s)
    if rc:
        raise RuntimeError("curl rc=%d:\n%s" % (rc, out[-400:]))
    return part


if WANT_LTX:
    for spec in LTX_SPECS:
        dest, want = spec["dest"], spec["bytes"]
        try:
            if dest.is_file() and dest.stat().st_size == want and not FORCE_REDOWNLOAD:
                rows.append((spec["label"], "%s  cached" % gb(want)))
                continue
            state["label"] = "downloading " + spec["label"]
            state["started"] = time.perf_counter()
            state["expected"] = want
            got = False
            problems = []
            for ref in spec["refs"]:
                try:
                    url = ref[1] if ref[0] == "url" else hf_find(ref[1], ref[2])
                    part = curl_resume(url, dest)
                    size = part.stat().st_size
                    if size != want:
                        raise RuntimeError("size mismatch: got %d, expected %d"
                                           % (size, want))
                    os.replace(part, dest)
                    rows.append((spec["label"], "%s  downloaded" % gb(want)))
                    got = True
                    break
                except Exception as exc:
                    problems.append("%s: %s" % (ref[1], str(exc)[:200]))
            if not got:
                raise RuntimeError(" | ".join(problems))
        except Exception as exc:
            errors.append("%s: %s" % (spec["label"], str(exc)[:260]))
            rows.append((spec["label"], "FAILED - see note"))

    present = [s for s in LTX_SPECS
               if s["dest"].is_file() and s["dest"].stat().st_size == s["bytes"]]
    LTX["models_ok"] = len(present) == len(LTX_SPECS) and not errors
else:
    LTX["models_ok"] = False
    rows.append(("LTX models", "skipped (MODEL_SET=%s)" % MODEL_SET))

ticker.stop_now()
if errors:
    live.finish("02 . models - incomplete", rows=rows, kind="err",
                note="<br>".join(errors) + "<br>Downloads resume on rerun. If a repo is "
                     "gated, set a HF_TOKEN in Colab secrets.")
elif WANT_H3 and WANT_LTX:
    live.finish("02 . models ready (H3 + LTX-2.5)", rows=rows, kind="ok",
                note="H3: DiT + VAEs + turbo LoRA%s. LTX: 22B DiT + gemma4 TE + both "
                     "VAEs + x2 upscaler. The H3 text encoder is %s; the Studio's Encode "
                     "caches conditioning in /content/h3lab/conditioning."
                     % (" + Ref2VA" if H3.get("ref2va_ok") else "",
                        "included" if need_te else "skipped"))
elif WANT_LTX:
    live.finish("02 . models ready (LTX-2.5)", rows=rows, kind="ok",
                note="MODEL_SET=ltx: only the LTX-2.5 stack was requested, so "
                     "every H3 file was left untouched. Set MODEL_SET to both "
                     "or h3 to fetch the H3 stack.")
else:
    live.finish("02 . models ready (H3)", rows=rows, kind="ok",
                note="MODEL_SET=h3: only the MiniMax H3 stack was requested%s, "
                     "so every LTX-2.5 file was left untouched. Set MODEL_SET "
                     "to both or ltx to fetch the LTX stack."
                     % (" (with Ref2VA)" if H3.get("ref2va_ok") else ""))


In [ ]:
# @title 03 . studio { display-mode: "form" }
# ComfyUI backend + combined H3 / LTX-2.5 Studio web UI (H3-Studio repo) behind a
# cloudflared tunnel. The checkout follows STUDIO_BRANCH and the UI is built from
# source here (npm), so what the tunnel serves always matches that branch. The cell
# prints only the tunnel URL and keeps running; stop it to shut everything down. The Studio's model switch picks MiniMax H3 or LTX-2.5;
# each job restarts ComfyUI only when the model family (and its fp16 flags) changes.
if "H3" not in globals():
    raise RuntimeError("run cell 00 first")
if "DIT_CHOICES" not in globals() or "LTX_FILES" not in globals():
    raise RuntimeError("run cell 02 first (model catalog)")
if not (COMFY_DIR / ".git").is_dir():
    raise RuntimeError("ComfyUI is not installed; run cell 01 first")

import hashlib, json, math, re, shutil, socket, sys, threading, time, uuid
from pathlib import Path
from urllib.parse import quote

import torch

try:
    import aiohttp  # noqa: F401
except ImportError:
    rc, out = sh([sys.executable, "-m", "pip", "install", "-q", "aiohttp"], timeout=900)
    if rc:
        raise RuntimeError("aiohttp install failed: " + out[-400:])

STUDIO_PORT = 8000
REPO_URL = "https://github.com/GodL-x-SouL/H3-Studio.git"
STUDIO_BRANCH = "testing"
REPO_DIR = WORK / "H3-Studio"
NODE_VERSION = "v22.14.0"   # Vite 8 needs node >= 20.19; Colab's is often older
CLOUDFLARED = WORK / "cloudflared"
UPLOAD_DIR = WORK / "uploads"
LORA_DIR = MODELS / "loras"
LORA_DIR.mkdir(parents=True, exist_ok=True)
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
FPS = 24
BASE_DIMS = {"0.3": (736, 416), "0.4": (864, 480)}
RATIOS = {"16:9": (16, 9), "9:16": (9, 16), "1:1": (1, 1), "4:3": (4, 3), "3:4": (3, 4)}
UNET_DTYPE_FLAGS = {"safe_fp16": "--force-fp16", "fp16_raw": "--force-fp16",
                    "bf16": "--bf16-unet", "fp16safe_node": None}
FIX_URL = ("https://raw.githubusercontent.com/Amduraznak/minimax-h3-fp16-fix/"
           "main/minimax_h3_fp16_fix.py")
SAGE_BASE = "https://github.com/netrunner-exe/SageAttention2-Colab-Wheels/raw/main"
SAGE_WHEELS = {
    ("cp312", "2.11.0", "128"): ("legacy_wheels", "colab-torch-cp312-2.11.0+cu128"),
    ("cp312", "2.11.0", "130"): ("legacy_wheels", "colab-torch-cp312-2.11.0+cu130"),
    ("cp312", "2.13.0", "130"): ("legacy_wheels", "colab-torch-cp312-2.13.0+cu130"),
    ("cp313", "2.11.0", "128"): ("optimized_wheels", "colab-torch-cp313-2.11.0+cu128"),
    ("cp313", "2.11.0", "130"): ("optimized_wheels", "colab-torch-cp313-2.11.0+cu130"),
    ("cp313", "2.13.0", "132"): ("optimized_wheels", "colab-torch-cp313-2.13.0+cu132"),
}
NODE_REPOS = {
    "fp16safe": ("FP16Safe", FP16SAFE_DIR,
                 "https://github.com/aaalll12322/ComfyUI-MiniMaxH3-FP16Safe.git"),
    "spectrum": ("Spectrum", SPECTRUM_DIR,
                 "https://github.com/xmarre/ComfyUI-Spectrum-MiniMax-H3.git"),
}


def _status(title, rows=None, kind="info", note="", details=""):
    return _box_html(title, rows or [], kind=kind, note=note, details=details)


def ratio_dims(gear, ratio):
    if ratio == "16:9":
        return BASE_DIMS[gear]
    budget = BASE_DIMS[gear][0] * BASE_DIMS[gear][1]
    rw, rh = RATIOS[ratio]
    width = max(32, round(math.sqrt(budget * rw / rh) / 32) * 32)
    height = max(32, round(math.sqrt(budget * rh / rw) / 32) * 32)
    return width, height


def _frame_sig(value):
    value = (value or "").strip()
    if not value:
        return None
    local = Path(value)
    if local.is_file():
        return hashlib.sha256(local.read_bytes()).hexdigest()
    return hashlib.sha256(("literal:" + value).encode("utf-8")).hexdigest()


def _clean_loras(value):
    """Normalise a LoRA stack: [{"name", "strength"}, ...]; legacy name/strength ok."""
    if isinstance(value, dict):
        value = [value]
    if isinstance(value, str):
        value = [{"name": value}]
    entries = []
    for item in value or []:
        if isinstance(item, dict):
            name = str(item.get("name") or "").strip()
            try:
                strength = float(item.get("strength", 1.0))
            except (TypeError, ValueError):
                strength = 1.0
        else:
            name, strength = str(item or "").strip(), 1.0
        if not name or name == "off":
            continue
        entries.append({"name": name, "strength": strength})
    return entries


def _lora_uses_comfy_keys(path):
    """True when the safetensors keys use ComfyUI's diffusion_model.* naming."""
    try:
        with open(path, "rb") as handle:
            header_size = int.from_bytes(handle.read(8), "little")
            if header_size <= 0 or header_size > 64 * 1024 * 1024:
                return False
            header = json.loads(handle.read(header_size).decode("utf-8"))
    except Exception:
        return False
    return any("diffusion_model." in key for key in header if key != "__metadata__")


# ------------------------------------------------------------- VRAM preflight --
# Peak VRAM measured on this stack (free T4, 15 GiB, ~120-frame clips):
#   H3  Q3_K_M 864x480 safe_fp16          -> 13.81 GiB peak
#   LTX 22B UD-Q2_K_XL 640x384 (x2 pass)  -> 12.4 GiB free before the refine step
# Both sit on one line: resident weights + ~0.10 GiB per megapixel-frame. The
# preflight refuses a geometry that cannot fit before ComfyUI restarts, loads the
# model or encodes a prompt, instead of OOMing an hour into the run.
VRAM_GIB_PER_MP_FRAME = 0.10
VRAM_RESERVE_GIB = 0.40
H3_RESIDENT_GIB = {"abiray_Q3_K_M": 8.85, "abiray_Q4_K_M": 11.33,
                   "ref2va_Q3_K_M": 8.85, "ref2va_Q4_K_M": 11.33}
LTX_RESIDENT_GIB = 9.33


def _gpu_vram_gib():
    """Total VRAM of device 0 in GiB; None when the runtime has no CUDA device."""
    if "_GPU_VRAM_GIB" not in globals():
        total = 0.0
        try:
            if torch.cuda.is_available():
                total = torch.cuda.get_device_properties(0).total_memory / 2 ** 30
        except Exception:
            total = 0.0
        globals()["_GPU_VRAM_GIB"] = total
    return globals()["_GPU_VRAM_GIB"] or None


def _gpu_label():
    try:
        if torch.cuda.is_available():
            return torch.cuda.get_device_name(0).strip()
    except Exception:
        pass
    return "this GPU"


def _vram_need_gib(weights_gib, width, height, frames):
    return weights_gib + VRAM_GIB_PER_MP_FRAME * (width * height / 1e6) * max(1, frames)


def _vram_preflight(label, weights_gib, width, height, frames, advice):
    """Raise before a run that the card cannot hold; no-op off CUDA."""
    total = _gpu_vram_gib()
    if total is None:
        return
    need = _vram_need_gib(weights_gib, width, height, frames)
    if need <= total - VRAM_RESERVE_GIB:
        return
    raise ValueError(
        "%s %dx%d at %d frames needs about %.1f GiB of VRAM (%.1f GiB weights + "
        "%.1f GiB activations); %s has %.1f GiB. %s"
        % (label, width, height, frames, need, weights_gib, need - weights_gib,
           _gpu_label(), total, advice))


def build_cfg(prompt, gear, ratio, length, steps, seed, dit_choice, loras,
              dtype_mode, use_sage, sage_mode, speed_method, text_branch,
              budget_min, triton, extra_flags, first_frame, last_frame,
              task="fl2v", references=None, ref_choice=None, ref_image_size="match",
              dims=None):
    prompt = (prompt or "").strip()
    if not prompt:
        raise ValueError("prompt is empty")
    length = int(length)
    if length < 5 or (length - 5) % 17 != 0:
        raise ValueError("length must satisfy n %% 17 == 5 (22, 39, ..., 124, ...); got %d"
                         % length)
    if ratio not in RATIOS:
        raise ValueError("unknown ratio %s" % ratio)
    task = "reference" if str(task or "").lower() == "reference" else "fl2v"
    if dims is not None:
        width, height = int(dims[0]), int(dims[1])
        if width < 256 or height < 256 or width % 32 or height % 32:
            raise ValueError("H3 width/height must be >= 256 and multiples of 32; "
                             "got %dx%d" % (width, height))
        if width > 2560 or height > 2560 or width * height > 2560 * 1440:
            raise ValueError("H3 above 2K (2560x1440) is not supported by this stack")
        gear = "0.4" if width * height >= 800 * 448 else "0.3"
    else:
        if gear not in BASE_DIMS:
            raise ValueError("unknown gear %s" % gear)
        width, height = ratio_dims(gear, ratio)
    steps = int(steps)
    warnings = []
    requested_loras = _clean_loras(loras)
    if task == "reference":
        ref_choice = ref_choice or globals().get("DEFAULT_REF2VA_CHOICE") or "Q3_K_M"
        if "REF2VA_CHOICES" not in globals() or ref_choice not in REF2VA_CHOICES:
            raise ValueError("Ref2VA model %r is not available; rerun cell 02 with "
                             "REF2VA_CHOICE set to Q3_K_M or Q4_K_M" % ref_choice)
        ref_images = [str(item).strip() for item in (references or [])
                      if str(item).strip()]
        if not ref_images:
            raise ValueError("Reference mode needs at least one reference image "
                             "(upload them in the Frames tab)")
        ref_images = ref_images[:9]
        dit = REF2VA_CHOICES[ref_choice]
        dit_choice = "ref2va_" + ref_choice
        lora_entries = []
    else:
        if dit_choice not in DIT_CHOICES:
            raise ValueError("unknown DiT choice %s" % dit_choice)
        dit = DIT_CHOICES[dit_choice]
        ref_images = []
        lora_entries = requested_loras
    _vram_preflight("H3", H3_RESIDENT_GIB.get(dit_choice, H3_RESIDENT_GIB["abiray_Q3_K_M"]),
                    width, height, length,
                    "Lower the resolution (Preview 864x480 is the tested T4 default), "
                    "shorten the clip, or use a runtime with more VRAM.")

    loras_cfg, seen = [], set()
    for entry in lora_entries:
        if entry["name"] in seen:
            raise ValueError("LoRA %s is listed twice; use one entry per LoRA and "
                             "set its strength once" % entry["name"])
        seen.add(entry["name"])
        lora_path = LORA_DIR / entry["name"]
        if not lora_path.is_file():
            raise ValueError("LoRA not found: %s (download it in the web UI)" % lora_path)
        loras_cfg.append({"name": entry["name"], "strength": float(entry["strength"]),
                          "is_turbo": "turbo" in entry["name"].lower(),
                          "loader": ("core" if _lora_uses_comfy_keys(lora_path)
                                     else "turbo"),
                          "bytes": lora_path.stat().st_size})
    lora_is_turbo = any(entry["is_turbo"] for entry in loras_cfg)
    dtype_flag = UNET_DTYPE_FLAGS[dtype_mode]
    use_sage_effective = bool(use_sage)
    if use_sage and dtype_flag != "--force-fp16":
        warnings.append("SageAttention needs fp16 activations; with %s the T4 falls back "
                        "to fp32, so Sage is disabled for this run. Use safe_fp16."
                        % dtype_mode)
        use_sage_effective = False
    if dtype_mode == "bf16":
        warnings.append("bf16 on T4 runs as fp32 compute: the slow baseline, ~342 s/step "
                        "at 864x480. Kept for comparison only.")
    if dtype_mode == "fp16_raw":
        warnings.append("fp16_raw is expected to produce NaN on H3 (condition_proj "
                        "overflow). Reference runs only.")
    if sum(1 for entry in loras_cfg if entry["loader"] == "turbo") > 1:
        warnings.append("Several H3 turbo-format LoRAs are chained through the turbo "
                        "node; stacking more than one can drop adapters. Use "
                        "ComfyUI-format LoRAs when you need a stack.")
    if dtype_mode == "fp16safe_node" and loras_cfg:
        warnings.append("fp16safe_node dequant-casts GGUF weights at runtime; watch "
                        "ComfyUI RSS in the Console tab. safe_fp16 remains the tested path.")
    if speed_method == "spectrum" and steps <= 6:
        warnings.append("Spectrum's win shrinks at low step counts; it may add little "
                        "at %d steps." % steps)
    if any(entry["is_turbo"] and "v4" in entry["name"] for entry in loras_cfg) \
            and steps <= 4:
        warnings.append("v4 at <=4 steps can smear heavy motion; the v1 turbo LoRA is the "
                        "safer 4-step choice.")
    if width * height > 1344 * 768:
        warnings.append("H3 at %dx%d (%.2f MP) is past the T4-tested envelope "
                        "(864x480); expect very slow steps or OOM. Lower the "
                        "resolution if the run dies."
                        % (width, height, width * height / 1e6))
    if dtype_mode != "bf16" and dit_choice == "abiray_Q4_K_M" \
            and width * height >= 864 * 480 and length > 56:
        warnings.append("Q4_K_M (10.77 GiB) + fp16-compute fp32 transients exceeds the "
                        "T4's ~14.5 GiB at %dx%d with long clips (verified OOM at "
                        "864x480/124f). Use abiray_Q3_K_M, a smaller resolution, or a "
                        "shorter length." % (width, height))
    if dtype_mode != "bf16" and task == "reference" and ref_choice == "Q4_K_M" \
            and width * height >= 864 * 480 and length > 56:
        warnings.append("Ref2VA Q4_K_M at %dx%d with long clips is likely to OOM "
                        "on a T4. Use Q3_K_M, a smaller resolution, or a shorter "
                        "length." % (width, height))
    if lora_is_turbo and steps > 10:
        warnings.append("Turbo LoRA is tuned for few steps; %d steps on top of it can "
                        "over-denoise. Choose LoRA 'off' for high step counts." % steps)
    if not loras_cfg and steps <= 8 and task != "reference":
        warnings.append("No LoRA: the base model usually wants more steps (12-20) than "
                        "the few-step LoRA profiles. steps=%d may look rough." % steps)
    if task == "reference":
        if requested_loras:
            warnings.append("Reference mode runs on the separate Ref2VA model; the FL2VA "
                            "LoRA stack (%s) was ignored."
                            % ", ".join(entry["name"] for entry in requested_loras))
        if steps < 12:
            warnings.append("Ref2VA has no turbo LoRA in this stack; 12-24 steps are "
                            "recommended (currently %d)." % steps)
        if ref_image_size not in ("match", "max"):
            ref_image_size = "match"
    if text_branch and dtype_mode == "safe_fp16":
        warnings.append("fp32 text branch is redundant with safe_fp16: the built-in fix "
                        "already upcasts condition_proj's input. Keep it off unless "
                        "diagnosing.")
    first_frame = (first_frame or "").strip() if task != "reference" else ""
    last_frame = (last_frame or "").strip() if task != "reference" else ""
    first_sig, last_sig = _frame_sig(first_frame), _frame_sig(last_frame)
    ref_sigs = [_frame_sig(item) for item in ref_images]
    key_payload = {
        "repo": MODEL_REPO, "revision": MODEL_REVISION,
        "text_encoder_sha256": TE_FILE["sha256"],
        "prompt_sha256": hashlib.sha256(prompt.encode("utf-8")).hexdigest(),
        "width": width, "height": height, "length": length,
        "task": task, "dit_sha256": dit["sha256"],
        "audio_vae_sha256": VAE_FILES["audio"]["sha256"],
    }
    if first_sig:
        key_payload["first_frame"] = first_sig
    if last_sig:
        key_payload["last_frame"] = last_sig
    if ref_sigs:
        key_payload["ref_images"] = ref_sigs
        key_payload["ref_image_size"] = ref_image_size
    key = "cond-" + hashlib.sha256(json.dumps(
        key_payload, sort_keys=True, separators=(",", ":")).encode("utf-8")).hexdigest()[:32]
    method_tag = []
    if use_sage_effective:
        method_tag.append("sage")
    if speed_method != "none":
        method_tag.append(speed_method)
    label = ("h3ref_" if task == "reference" else "h3_") + "%sstep_%dx%d_%df_%s%s" % (
        steps, width, height, length, dtype_mode,
        ("_" + "_".join(method_tag)) if method_tag else "")
    chain = "UnetLoaderGGUF(%s)" % ("Ref2VA" if task == "reference" else dit_choice)
    for entry in loras_cfg:
        chain += " -> %s(%s@%g)" % ("LoRA" if entry["loader"] == "core" else "TurboLoRA",
                                    entry["name"], entry["strength"])
    if text_branch:
        chain += " -> Fp32TextBranch"
    if dtype_mode == "fp16safe_node":
        chain += " -> FP16Safe"
    if speed_method == "spectrum":
        chain += " -> Spectrum"
    if use_sage_effective:
        chain += " -> Sage(%s)" % sage_mode
    if loras_cfg:
        chain += " -> %s" % ("MiniMaxH3TurboSampler" if lora_is_turbo
                             else "KSamplerSelect(euler)")
    cfg = {
        "prompt": prompt, "width": width, "height": height, "length": length, "fps": FPS,
        "gear": gear, "ratio": ratio, "steps": steps, "seed": int(seed),
        "dit": dit, "dit_choice": dit_choice,
        "task": task, "references": ref_images, "ref_image_size": ref_image_size,
        "sampler_name": "res_multistep" if task == "reference" else "euler",
        "loras": loras_cfg, "lora_is_turbo": lora_is_turbo,
        "dtype_mode": dtype_mode, "dtype_flag": dtype_flag,
        "use_sage": use_sage_effective, "sage_mode": sage_mode,
        "speed_method": speed_method,
        "text_branch": bool(text_branch), "triton": bool(triton),
        "extra_flags": (extra_flags or "").strip(),
        "first_frame": first_frame, "last_frame": last_frame,
        "budget_s": int(budget_min) * 60, "window_s": 120, "max_windows": 2,
        "sampler_start_timeout_s": 900,
        "key": key, "label": label, "latent_key": label,
        "video_prefix": "h3_lab/" + label, "expected_shapes": [
            [1, 24, ((length - 5) // 17) * 5 + 2, height // 16, width // 16],
            [1, 32, 2, round(length / FPS * 40)]],
        "te_file": TE_FILE, "video_vae": VAE_FILES["video"], "audio_vae": VAE_FILES["audio"],
        "blob_paths": [str(WORK / "conditioning" / (key + ".pt"))],
        "chain": chain, "warnings": warnings,
    }
    return cfg


def _payload_to_cfg(data):
    frames = int(data.get("frames") or 124)
    frames = max(22, min(277, ((frames - 5) // 17) * 17 + 5))
    scale = int(data.get("scale") or 100)
    gear = "0.4" if scale >= 80 else "0.3"
    width = int(data.get("width") or 0)
    height = int(data.get("height") or 0)
    dims = (width, height) if width >= 256 and height >= 256 else None
    speed = {"none": "none", "4-step": "none",
             "Spectrum": "spectrum"}.get(str(data.get("speed") or "none"), "none")
    mode = str(data.get("mode") or "text").lower()
    references = data.get("references") or []
    if isinstance(references, str):
        references = [references]
    if mode != "reference":
        references = []
    return build_cfg(
        str(data.get("prompt") or ""), gear, str(data.get("ratio") or "16:9"), frames,
        int(data.get("steps") or 6), int(data.get("seed") or 0),
        str(data.get("dit_choice") or "abiray_Q3_K_M"),
        data.get("loras") if isinstance(data.get("loras"), list)
        else [{"name": data.get("lora"), "strength": data.get("strength")}],
        str(data.get("dtype") or "safe_fp16"), bool(data.get("sage", True)),
        str(data.get("sage_mode") or "sageattn_qk_int8_pv_fp16_cuda"), speed,
        False,
        float(data.get("budget") or 75), bool(data.get("triton", True)),
        str(data.get("extra_flags") or ""),
        str(data.get("first_frame") or "") if mode == "image" else "",
        str(data.get("last_frame") or "") if mode == "image" else "",
        task="reference" if mode == "reference" else "fl2v",
        references=references,
        ref_choice=str(data.get("ref_choice") or "") or None,
        ref_image_size=str(data.get("ref_image_size") or "match"),
        dims=dims)

def _lora_choices():
    return ["off"] + sorted(p.name for p in LORA_DIR.glob("*.safetensors"))


def _default_lora():
    preferred = TURBO_LORA_CHOICES["v4_step600_ema"]["file"]
    return preferred if (LORA_DIR / preferred).is_file() else "off"


def download_lora(url):
    url = (url or "").strip()
    if not url:
        raise ValueError("paste a HuggingFace or direct .safetensors URL first")
    try:
        from huggingface_hub import hf_hub_download
    except ImportError:
        rc, out = sh([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"],
                     timeout=900)
        if rc:
            raise RuntimeError("huggingface_hub install failed: " + out[-400:])
        from huggingface_hub import hf_hub_download
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass
    target = None
    match = re.match(
        r"https?://huggingface\.co/([^/]+/[^/]+)/(?:blob|resolve)/([^/]+)/(.+)", url)
    if match:
        repo_id, revision, filename = match.groups()
        filename = filename.split("?")[0].strip()
        if not filename.endswith(".safetensors"):
            raise ValueError("only .safetensors LoRAs are supported")
        target = Path(hf_hub_download(repo_id=repo_id, filename=filename,
                                      revision=revision, local_dir=str(LORA_DIR),
                                      token=token))
    elif url.startswith(("http://", "https://")) and ".safetensors" in url:
        name = url.split("?")[0].rstrip("/").split("/")[-1]
        if not name.endswith(".safetensors"):
            raise ValueError("direct links must end in .safetensors")
        target = LORA_DIR / name
        if not target.is_file() or target.stat().st_size < 1_000_000:
            with requests.get(url, stream=True, timeout=1800) as response:
                response.raise_for_status()
                staged = target.with_suffix(".part")
                with staged.open("wb") as handle:
                    for chunk in response.iter_content(1024 * 1024):
                        handle.write(chunk)
                staged.replace(target)
    else:
        raise ValueError("expected https://huggingface.co/<repo>/blob/<rev>/<file>"
                         ".safetensors")
    if target.parent != LORA_DIR:
        moved = LORA_DIR / target.name
        shutil.copy2(target, moved)
        target = moved
    return target.name


def _download_fp16_fix():
    response = requests.get(FIX_URL, timeout=60)
    response.raise_for_status()
    text = response.text
    for needle in ("K_OUT_PROJ", "K_FC2", "condition_proj"):
        if needle not in text:
            raise RuntimeError("downloaded file does not look like the fp16 fix "
                               "(missing %s)" % needle)
    FP16_FIX_FILE.write_text(text, encoding="utf-8")
    return hashlib.sha256(text.encode()).hexdigest()


def _ensure_fp16_fix(mode):
    if mode == "safe_fp16":
        if not FP16_FIX_FILE.is_file():
            digest = _download_fp16_fix()
            H3["fp16_patch"] = True
            return "downloaded + active (sha %s)" % digest[:12]
        H3["fp16_patch"] = True
        return "active"
    existed = FP16_FIX_FILE.is_file()
    if existed:
        FP16_FIX_FILE.unlink()
    H3["fp16_patch"] = False
    return "removed for %s" % mode if existed else "off for %s" % mode


def sync_fp16_fix(active):
    if active:
        digest = _download_fp16_fix()
        H3["fp16_patch"] = True
        return "active (sha %s)" % digest[:12]
    existed = FP16_FIX_FILE.is_file()
    if existed:
        FP16_FIX_FILE.unlink()
    H3["fp16_patch"] = False
    return "removed" if existed else "not installed"


def _verify_node(kind):
    if kind == "fp16safe":
        path = FP16SAFE_DIR / "nodes.py"
        return path.is_file() and "MiniMaxH3FP16Safe" in path.read_text(
            encoding="utf-8", errors="ignore")
    for path in SPECTRUM_DIR.rglob("*.py"):
        try:
            if "SpectrumApplyMiniMaxH3" in path.read_text(encoding="utf-8",
                                                          errors="ignore"):
                return True
        except Exception:
            pass
    return False


def sync_node(kind, active):
    if not active:
        return "off"
    label, dirpath, url = NODE_REPOS[kind]
    if not (dirpath / ".git").is_dir():
        rc, out = sh(["git", "clone", "--depth", "1", url, str(dirpath)], timeout=600)
        if rc:
            raise RuntimeError("%s clone failed: %s" % (label, out[-400:]))
    req = dirpath / "requirements.txt"
    if req.is_file():
        rc, out = sh([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)],
                     timeout=900)
        if rc:
            raise RuntimeError("%s deps failed: %s" % (label, out[-400:]))
    if not _verify_node(kind):
        raise RuntimeError("%s node class not found after clone; repo layout changed"
                           % label)
    return "installed"


def setup_state():
    return {
        "fp16fix": FP16_FIX_FILE.is_file(),
        "fp16safe": (FP16SAFE_DIR / ".git").is_dir(),
        "spectrum": (SPECTRUM_DIR / ".git").is_dir(),
        "sage": bool(H3.get("sage_ok")),
    }


def apply_setup(flags):
    flags = flags or {}
    rows = []
    for label, action in (
            ("fp16 NaN fix", lambda: sync_fp16_fix(bool(flags.get("fp16fix")))),
            ("FP16Safe", lambda: sync_node("fp16safe", bool(flags.get("fp16safe")))),
            ("Spectrum", lambda: sync_node("spectrum", bool(flags.get("spectrum"))))):
        try:
            rows.append((label, action()))
        except Exception as exc:
            rows.append((label, "FAILED: %s" % str(exc)[:200]))
    rows.append(("SageAttention", "import ok" if H3.get("sage_ok") else "not installed"))
    return rows


def install_sage():
    py_tag = "cp%d%d" % (sys.version_info.major, sys.version_info.minor)
    torch_tag = torch.__version__.split("+")[0].split(".dev")[0]
    cuda_tag = (torch.version.cuda or "").replace(".", "")
    key = (py_tag, torch_tag, cuda_tag)
    rows = [("runtime", "python %s, torch %s, cuda %s"
             % (py_tag, torch_tag, cuda_tag))]
    if key not in SAGE_WHEELS:
        rows.append(("state", "no prebuilt wheel for this runtime"))
        return rows
    subdir, folder = SAGE_WHEELS[key]
    filename = "sageattention-2.1.2-%s-%s-linux_x86_64.whl" % (py_tag, py_tag)
    url = "%s/%s/%s/%s" % (SAGE_BASE, subdir, folder, filename)
    wheel = WHEEL_DIR / filename
    try:
        if not wheel.is_file() or wheel.stat().st_size < 1_000_000:
            response = requests.get(url, timeout=600)
            response.raise_for_status()
            wheel.write_bytes(response.content)
        rows.append(("wheel", "%s (%.1f MiB)" % (filename, wheel.stat().st_size / 2**20)))
        rc, out = sh([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                      str(wheel)], timeout=600)
        if rc:
            raise RuntimeError("pip install failed: " + out[-400:])
        rc, out = sh([sys.executable, "-c",
                      "import sageattention; print(getattr(sageattention, "
                      "'__version__', '?'))"])
        if rc:
            raise RuntimeError("import check failed: " + out[-400:])
        H3["sage_ok"] = True
        rows.append(("sageattention", out.strip()))
    except Exception as exc:
        H3["sage_ok"] = False
        rows.append(("state", "FAILED: %s" % str(exc)[:200]))
    return rows


try:
    import sageattention as _sage_probe
    H3["sage_ok"] = True
except Exception:
    H3["sage_ok"] = False
H3.setdefault("fp16_patch", FP16_FIX_FILE.is_file())
H3.setdefault("studio_last_ok", None)


def _sniff_ext(data):
    if data[:8] == b"\x89PNG\r\n\x1a\n":
        return ".png"
    if data[:2] == b"\xff\xd8":
        return ".jpg"
    if data[:4] == b"RIFF" and data[8:12] == b"WEBP":
        return ".webp"
    if data[:2] == b"BM":
        return ".bmp"
    return ".png"


def prepare_image(value, tag):
    value = (value or "").strip()
    if not value:
        return None
    src = Path(value)
    if src.is_file():
        data = src.read_bytes()
    elif value.lower().startswith(("http://", "https://")):
        response = requests.get(value, timeout=180)
        response.raise_for_status()
        data = response.content
    else:
        raise RuntimeError("%s_frame not found: %s (upload it in the Frames tab)"
                           % (tag, value))
    if len(data) < 512:
        raise RuntimeError("%s_frame image is suspiciously small (%d bytes)"
                           % (tag, len(data)))
    name = "h3lab_%s_%s%s" % (tag, hashlib.sha256(data).hexdigest()[:12], _sniff_ext(data))
    input_dir = COMFY_DIR / "input"
    input_dir.mkdir(parents=True, exist_ok=True)
    (input_dir / name).write_bytes(data)
    return name


def _stage_upload(filename, data):
    data = bytes(data)
    if len(data) < 512:
        raise RuntimeError("uploaded image is suspiciously small (%d bytes)" % len(data))
    ext = _sniff_ext(data)
    dest = UPLOAD_DIR / ("h3lab_%s%s" % (hashlib.sha256(data).hexdigest()[:12], ext))
    if not dest.is_file():
        dest.write_bytes(data)
    return str(dest)


def ensure_server(cfg, force=False):
    want = cfg["dtype_flag"]
    extra = (["--enable-triton-backend"] if cfg["triton"] else []) \
        + (cfg["extra_flags"].split() if cfg["extra_flags"] else [])
    stats = None
    try:
        stats = requests.get(BASE_URL + "/system_stats", timeout=5).json()
    except Exception:
        stats = None
    if stats is not None and not force:
        argv = stats.get("system", {}).get("argv") or []
        mismatch = H3.get("server_family") != "h3"
        if want and want not in argv:
            mismatch = True
        if want is None and ("--force-fp16" in argv or "--bf16-unet" in argv):
            mismatch = True
        if cfg["triton"] and "--enable-triton-backend" not in argv:
            mismatch = True
        if not mismatch:
            # Start every stage at the current end of the log. Reusing the last 20 KB
            # of an older job would feed stale tqdm percentages to the progress bar.
            offset = COMFY_LOG.stat().st_size if COMFY_LOG.exists() else 0
            return stats, offset
    stats, offset = start_comfy(want, extra=extra)
    H3["server_family"] = "h3"
    return stats, offset


def _log_tail_text():
    if not COMFY_LOG.exists():
        return "(no log yet)"
    text = COMFY_LOG.read_bytes()[-60000:].decode("utf-8", errors="replace")
    lines = [l for l in text.splitlines() if l.strip()]
    return "\n".join(lines[-150:])


def op_encode(cfg, keep_server):
    key = cfg["key"]
    store = WORK / "conditioning"
    store.mkdir(parents=True, exist_ok=True)
    dest = store / (key + ".pt")
    task = cfg.get("task") or "fl2v"
    if dest.is_file() and dest.stat().st_size > 1024 * 1024:
        H3["blob_ready"] = True
        H3["studio_last_ok"] = True
        yield (_status("Encode recalled", [
            ("blob", str(dest)), ("size", gb(dest.stat().st_size)), ("key", key),
            ("task", task)], kind="ok",
            note="Conditioning for this prompt/geometry/references already exists; "
                 "nothing to encode."), "", None)
        return
    te_path = MODELS / cfg["te_file"]["path"]
    if not te_path.is_file():
        H3["studio_last_ok"] = False
        yield (_status("Encode blocked", [
            ("text encoder", "missing at %s" % te_path),
            ("fix", "run cell 02 with the text encoder included")], kind="err"),
            "", None)
        return
    rows, errors, log_lines = [
        ("key", key), ("task", "reference (Ref2VA)" if task == "reference" else "FL2VA"),
        ("geometry", "%dx%d / %d frames" % (cfg["width"], cfg["height"],
                                             cfg["length"]))], [], []
    started = time.perf_counter()
    try:
        image_names, ref_names = {}, []
        if task == "reference":
            for index, value in enumerate(cfg.get("references") or []):
                name = prepare_image(value, "reference image %d" % (index + 1))
                if name:
                    ref_names.append((index, name))
            if ref_names:
                rows.append(("references", ", ".join(
                    "<Picture %d>=%s" % (index + 1, name)
                    for index, name in ref_names)))
        else:
            for tag, slot in (("first_frame", "first_frame"),
                              ("last_frame", "last_frame")):
                name = prepare_image(cfg.get(slot, ""), tag)
                if name:
                    image_names[tag] = name
            if image_names:
                rows.append(("images", ", ".join("%s=%s" % kv
                                                 for kv in sorted(image_names.items()))))
        yield (_status("Encode", rows, kind="run",
                       note="starting ComfyUI / staging inputs ..."), "", None)
        stats, log_offset = ensure_server(cfg, force=not keep_server)
        info = get_object_info()
        H3["object_info"] = info
        if task == "reference":
            required = ["CLIPLoader", "VAELoader", "MiniMaxH3ReferenceToVideo",
                        "H3SaveConditioning"] + (["LoadImage"] if ref_names else [])
            for node in required:
                if node not in info:
                    raise RuntimeError("%s missing from object_info; rerun cell 01"
                                       % node)
            graph = {
                "2": {"class_type": "CLIPLoader", "inputs": {
                    "clip_name": cfg["te_file"]["name"], "type": "minimax",
                    "device": "default"}},
                "3": {"class_type": "VAELoader",
                      "inputs": {"vae_name": cfg["video_vae"]["name"]}},
                "4": {"class_type": "VAELoader",
                      "inputs": {"vae_name": cfg["audio_vae"]["name"]}},
                "5": {"class_type": "MiniMaxH3ReferenceToVideo", "inputs": {
                    "clip": ["2", 0], "vae": ["3", 0], "audio_vae": ["4", 0],
                    "prompt": cfg["prompt"], "width": cfg["width"],
                    "height": cfg["height"], "length": cfg["length"],
                    "ref_image_size": cfg.get("ref_image_size") or "match"}},
                "40": {"class_type": "H3SaveConditioning", "inputs": {
                    "conditioning": ["5", 0], "samples": ["5", 1],
                    "store_dir": str(store), "key": key}},
            }
            for index, name in ref_names:
                node_id = str(100 + index)
                graph[node_id] = {"class_type": "LoadImage",
                                  "inputs": {"image": name}}
                graph["5"]["inputs"]["ref_images.ref_image_%d" % index] = [node_id, 0]
        else:
            required = ["CLIPLoader", "VAELoader", "MiniMaxH3ImageToVideo",
                        "H3SaveConditioning"] + (["LoadImage"] if image_names else [])
            for node in required:
                if node not in info:
                    raise RuntimeError("%s missing from object_info; rerun cell 01"
                                       % node)
            node_spec = info["MiniMaxH3ImageToVideo"].get("input", {})
            node_inputs = (set(node_spec.get("required", {}))
                           | set(node_spec.get("optional", {})))
            for tag, slot in (("first_frame", "first_frame"),
                              ("last_frame", "last_frame")):
                if tag in image_names and slot not in node_inputs:
                    raise RuntimeError("this ComfyUI build's MiniMaxH3ImageToVideo has no "
                                       "%s input (schema: %s)"
                                       % (slot, sorted(node_inputs)))
            graph = {
                "2": {"class_type": "CLIPLoader", "inputs": {
                    "clip_name": cfg["te_file"]["name"], "type": "minimax",
                    "device": "default"}},
                "3": {"class_type": "VAELoader",
                      "inputs": {"vae_name": cfg["video_vae"]["name"]}},
                "5": {"class_type": "MiniMaxH3ImageToVideo", "inputs": {
                    "clip": ["2", 0], "vae": ["3", 0], "prompt": cfg["prompt"],
                    "width": cfg["width"], "height": cfg["height"],
                    "length": cfg["length"]}},
                "40": {"class_type": "H3SaveConditioning", "inputs": {
                    "conditioning": ["5", 0], "samples": ["5", 1],
                    "store_dir": str(store), "key": key}},
            }
            if "first_frame" in image_names:
                graph["0"] = {"class_type": "LoadImage",
                              "inputs": {"image": image_names["first_frame"]}}
                graph["5"]["inputs"]["first_frame"] = ["0", 0]
            if "last_frame" in image_names:
                graph["00"] = {"class_type": "LoadImage",
                               "inputs": {"image": image_names["last_frame"]}}
                graph["5"]["inputs"]["last_frame"] = ["00", 0]
        problems = validate_graph(graph, info)
        if problems:
            raise RuntimeError("encode graph invalid:\n" + "\n".join(problems))
        prompt_id = str(uuid.uuid4())
        shown = log_offset
        response = requests.post(BASE_URL + "/prompt", timeout=120, json={
            "prompt": graph, "client_id": str(uuid.uuid4()), "prompt_id": prompt_id})
        response.raise_for_status()
        if response.json().get("node_errors"):
            raise RuntimeError("Node errors: %s" % response.json()["node_errors"])
        record = None
        while time.perf_counter() - started < 3600:
            proc = H3.get("proc")
            if proc is not None and proc.poll() is not None:
                raise RuntimeError("ComfyUI exited during encode (OOM kill?)")
            chunk = tail_comfy_log(shown, 500)
            shown = COMFY_LOG.stat().st_size if COMFY_LOG.exists() else shown
            if chunk:
                log_lines.extend(chunk.splitlines())
            progress = latest_progress(chunk)
            if progress:
                status_line = "%s  %d/%d" % (
                    pbar(progress["steps_completed"] / max(1, progress["steps_total"])),
                    progress["steps_completed"], progress["steps_total"])
            else:
                lines = [l for l in chunk.splitlines() if l.strip()]
                status_line = (lines[-1][:150].replace("<", "&lt;") if lines
                               else "(initializing)")
            yield (_status("Encode", rows, kind="run",
                           note="elapsed %s<br>%s" % (fmt_secs(time.perf_counter()
                                                                - started), status_line)),
                   "\n".join(log_lines[-200:]), None)
            try:
                history = requests.get(BASE_URL + "/history/" + prompt_id,
                                       timeout=60).json()
                if prompt_id in history:
                    status = history[prompt_id].get("status", {})
                    if status.get("completed") is True \
                            or status.get("status_str") == "error":
                        record = history[prompt_id]
                        break
            except Exception:
                pass
            time.sleep(3)
        if record is None:
            raise TimeoutError("encode did not finish within 3600s")
        if record.get("status", {}).get("status_str") != "success":
            raise RuntimeError("encode failed: %s" % record.get("status"))
        rows += [("status", "encoded in %.0fs" % (time.perf_counter() - started)),
                 ("blob", "%s (%s)" % (dest, gb(dest.stat().st_size)))]
        summary_path = store / (key + ".summary.json")
        if summary_path.is_file():
            summary = json.loads(summary_path.read_text(encoding="utf-8"))
            rows.append(("shapes", json.dumps(
                summary.get("samples", {}).get("parts", []))[:120].replace("\\n", " ")))
        H3["blob_ready"] = True
    except Exception as exc:
        errors.append("%s: %s" % (type(exc).__name__, str(exc)[:400]))
    finally:
        if not keep_server:
            stop_comfy()
    if errors:
        H3["studio_last_ok"] = False
        yield (_status("Encode FAILED", rows, kind="err", note="<br>".join(errors)),
               "\n".join(log_lines[-200:]), None)
    else:
        H3["studio_last_ok"] = True
        yield (_status("Encode done", rows, kind="ok",
                       note="ComfyUI stopped to free RAM." if not keep_server
                       else "ComfyUI left running for the next step."),
               "\n".join(log_lines[-200:]), None)

def op_sample(cfg, keep_server):
    key = cfg["key"]
    latent_dir = WORK / "latents"
    latent_dir.mkdir(parents=True, exist_ok=True)
    cond_blob = WORK / "conditioning" / (key + ".pt")
    if not cond_blob.is_file():
        H3["studio_last_ok"] = False
        yield (_status("Sample blocked", [("conditioning", "missing: %s" % cond_blob),
                                          ("fix", "run Encode first")], kind="err"),
               "", None)
        return
    if cfg["use_sage"] and not H3.get("sage_ok"):
        H3["studio_last_ok"] = False
        yield (_status("Sample blocked", [
            ("sage", "wheel not installed"),
            ("fix", "Setup tab: Install SageAttention wheel, or turn Use Sage off")],
            kind="err"), "", None)
        return
    rows, errors, warnings, log_lines = [], [], [], []
    stop_watch = None
    started = time.perf_counter()
    try:
        rows.append(("fp16 fix", _ensure_fp16_fix(cfg["dtype_mode"])))
        stats, log_offset = ensure_server(cfg, force=not keep_server)
        info = get_object_info()
        H3["object_info"] = info
        needed = {"UnetLoaderGGUF", "H3LoadConditioning", "RandomNoise", "BasicScheduler",
                  "BasicGuider", "SamplerCustomAdvanced", "KSamplerSelect", "H3SaveLatent",
                  "H3LatentProbe"}
        for entry in cfg["loras"]:
            needed.add("LoraLoaderModelOnly" if entry["loader"] == "core"
                       else "MiniMaxH3TurboLoRA")
        if cfg["loras"] and cfg["lora_is_turbo"]:
            needed.add("MiniMaxH3TurboSampler")
        if cfg["dtype_mode"] == "fp16safe_node":
            needed.add("MiniMaxH3FP16Safe")
        if cfg["speed_method"] == "spectrum":
            needed.add("SpectrumApplyMiniMaxH3")
        missing = sorted(n for n in needed if n not in info)
        if missing:
            raise RuntimeError("nodes not loaded: %s - install them in the Setup tab, then "
                               "run again" % ", ".join(missing))
        model_src = ["1", 0]
        extra_nodes = {}
        for index, entry in enumerate(cfg["loras"]):
            node_id = "lora%d" % index
            if entry["loader"] == "core":
                extra_nodes[node_id] = {"class_type": "LoraLoaderModelOnly", "inputs": {
                    "model": model_src, "lora_name": entry["name"],
                    "strength_model": entry["strength"]}}
            else:
                extra_nodes[node_id] = {"class_type": "MiniMaxH3TurboLoRA", "inputs": {
                    "model": model_src, "lora_name": entry["name"],
                    "strength": entry["strength"], "low_vram": False}}
            model_src = [node_id, 0]
        if cfg["text_branch"]:
            extra_nodes["17"] = {"class_type": "H3Fp32TextBranch",
                                 "inputs": {"model": model_src}}
            model_src = ["17", 0]
        if cfg["dtype_mode"] == "fp16safe_node":
            extra_nodes["14"] = {"class_type": "MiniMaxH3FP16Safe",
                                 "inputs": {"model": model_src}}
            model_src = ["14", 0]
        if cfg["speed_method"] == "spectrum":
            extra_nodes["22"] = {"class_type": "SpectrumApplyMiniMaxH3", "inputs": {
                "model": model_src, "enabled": True, "blend_weight": 0.5, "degree": 1,
                "ridge_lambda": 0.1, "window_size": 2.0, "flex_window": 0.75,
                "warmup_steps": 1, "tail_actual_steps": 1, "max_history": 8,
                "debug": False}}
            model_src = ["22", 0]
        if cfg["use_sage"]:
            extra_nodes["20"] = {"class_type": "H3SagePatch",
                                 "inputs": {"model": model_src,
                                            "mode": cfg["sage_mode"]}}
            model_src = ["20", 0]
        graph = {
            "1": {"class_type": "UnetLoaderGGUF",
                  "inputs": {"unet_name": cfg["dit"]["file"]}},
            "41": {"class_type": "H3LoadConditioning", "inputs": {
                "store_dir": str(WORK / "conditioning"), "key": key}},
            "6": {"class_type": "RandomNoise", "inputs": {"noise_seed": cfg["seed"]}},
            "7": {"class_type": "BasicScheduler", "inputs": {
                "model": model_src, "scheduler": "simple", "steps": cfg["steps"],
                "denoise": 1.0}},
            "8": ({"class_type": "MiniMaxH3TurboSampler", "inputs": {}}
                  if (cfg["loras"] and cfg["lora_is_turbo"])
                  else {"class_type": "KSamplerSelect",
                        "inputs": {"sampler_name": cfg.get("sampler_name") or "euler"}}),
            "9": {"class_type": "BasicGuider", "inputs": {
                "model": model_src, "conditioning": ["41", 0]}},
            "10": {"class_type": "SamplerCustomAdvanced", "inputs": {
                "noise": ["6", 0], "guider": ["9", 0], "sampler": ["8", 0],
                "sigmas": ["7", 0], "latent_image": ["41", 1]}},
            "18": {"class_type": "H3SaveLatent", "inputs": {
                "samples": ["10", 0], "store_dir": str(latent_dir),
                "key": cfg["latent_key"]}},
            "15": {"class_type": "H3LatentProbe", "inputs": {
                "samples": ["18", 0], "filename_prefix": cfg["video_prefix"]}},
        }
        graph.update(extra_nodes)
        problems = validate_graph(graph, info)
        if problems:
            raise RuntimeError("sample graph invalid: " + "; ".join(problems))
        (WORK / "sample_graph.json").write_text(json.dumps(graph, indent=2),
                                                encoding="utf-8")
        rows += [("graph", ", ".join(sorted(n["class_type"]
                                            for n in graph.values()))[:200]),
                 ("sampler", "MiniMaxH3TurboSampler"
                  if (cfg["loras"] and cfg["lora_is_turbo"])
                  else "KSamplerSelect (%s)" % (cfg.get("sampler_name") or "euler"))]
        prompt_id = str(uuid.uuid4())
        shown = log_offset
        maxima = {"rss_gib": 0.0, "vram_mib": 0.0, "host_low_gib": 1e9}
        stop_watch = threading.Event()

        def watch():
            tick = 0
            while not stop_watch.is_set():
                try:
                    host = psutil.virtual_memory().available / GIB
                    maxima["host_low_gib"] = min(maxima["host_low_gib"], host)
                    proc = H3.get("proc")
                    if proc is not None:
                        try:
                            rss = psutil.Process(proc.pid).memory_info().rss
                            for child in psutil.Process(proc.pid).children(recursive=True):
                                try:
                                    rss += child.memory_info().rss
                                except Exception:
                                    pass
                            maxima["rss_gib"] = max(maxima["rss_gib"], rss / GIB)
                        except Exception:
                            pass
                    if tick % 2 == 0:
                        rc, out = sh(["nvidia-smi", "--query-gpu=memory.used",
                                      "--format=csv,noheader,nounits"], timeout=10)
                        if rc == 0:
                            try:
                                maxima["vram_mib"] = max(
                                    maxima["vram_mib"],
                                    float(out.strip().splitlines()[0]))
                            except Exception:
                                pass
                except Exception:
                    pass
                tick += 1
                stop_watch.wait(1.0)

        watcher = threading.Thread(target=watch, daemon=True)
        watcher.start()

        def cancel(pid):
            try:
                requests.post("%s/api/jobs/%s/cancel" % (BASE_URL, pid), timeout=30)
            except Exception:
                try:
                    requests.post(BASE_URL + "/interrupt", timeout=30)
                except Exception:
                    pass

        window_s, budget_s = cfg["window_s"], cfg["budget_s"]
        guard = {"outcome": None, "windows": 0, "sampler_started_at": None,
                 "settled": False, "projected": None, "observed_s_per_step": None}
        response = requests.post(BASE_URL + "/prompt", timeout=120, json={
            "prompt": graph, "client_id": str(uuid.uuid4()), "prompt_id": prompt_id})
        response.raise_for_status()
        if response.json().get("node_errors"):
            raise RuntimeError("Node errors: %s" % response.json()["node_errors"])
        rows.append(("submitted", "%d steps @ %dx%d / %d frames"
                     % (cfg["steps"], cfg["width"], cfg["height"], cfg["length"])))
        deadline = started + cfg["sampler_start_timeout_s"] + budget_s + 1800
        record = None
        while time.perf_counter() < deadline:
            proc = H3.get("proc")
            if proc is not None and proc.poll() is not None:
                raise RuntimeError("ComfyUI exited during sampling (OOM kill?)")
            new = tail_comfy_log(shown, 500)
            shown = COMFY_LOG.stat().st_size if COMFY_LOG.exists() else shown
            if new:
                log_lines.extend(new.splitlines())
            try:
                history = requests.get(BASE_URL + "/history/" + prompt_id,
                                       timeout=60).json()
                if prompt_id in history:
                    status = history[prompt_id].get("status", {})
                    if status.get("completed") is True \
                            or status.get("status_str") == "error":
                        record = history[prompt_id]
                        break
            except Exception:
                pass
            now = time.perf_counter()
            progress = latest_progress("\n".join(log_lines[-400:]))
            if progress is None:
                yield (_status("Sampling", rows, kind="run",
                               note="loading model / waiting for the sampler to start · "
                                    "elapsed %s" % fmt_secs(now - started)),
                       "\n".join(log_lines[-200:]), None)
                if now - started > cfg["sampler_start_timeout_s"]:
                    guard["outcome"] = "SAMPLER_NEVER_STARTED"
                    cancel(prompt_id)
                    break
            else:
                if guard["sampler_started_at"] is None:
                    guard["sampler_started_at"] = now - progress["elapsed_in_loop_s"]
                observed = now - guard["sampler_started_at"]
                if not guard["settled"]:
                    if progress["steps_completed"] <= 0:
                        no_step_deadline = max(window_s * cfg["max_windows"],
                                               2.0 * budget_s / max(cfg["steps"], 1))
                        if observed >= no_step_deadline:
                            guard["outcome"] = "NO_STEP_COMPLETED"
                            cancel(prompt_id)
                            break
                    elif observed >= window_s * (guard["windows"] + 1):
                        guard["windows"] += 1
                        guard["observed_s_per_step"] = progress["observed_s_per_step"]
                        guard["projected"] = progress["observed_s_per_step"] * cfg["steps"]
                        if guard["projected"] > budget_s:
                            guard["outcome"] = "PROJECTED_OVER_BUDGET"
                            cancel(prompt_id)
                            break
                        guard["settled"] = True
                s_per_step = progress["observed_s_per_step"]
                remaining = (s_per_step * (cfg["steps"] - progress["steps_completed"])
                             if s_per_step else None)
                live_rows = rows + [
                    ("sampling", "%s  step %d/%d" % (
                        pbar(progress["steps_completed"]
                             / max(1, progress["steps_total"])),
                        progress["steps_completed"], progress["steps_total"])),
                    ("s/step", ("%.1f" % s_per_step) if s_per_step else "measuring ..."),
                    ("eta", fmt_secs(remaining) if remaining else "?"),
                    ("sampling elapsed", fmt_secs(observed)),
                    ("VRAM", "%.0f MiB" % maxima["vram_mib"]),
                    ("guard", "%d/%d windows" % (guard["windows"], cfg["max_windows"])),
                ]
                yield (_status("Sampling", live_rows, kind="run",
                               note="working · elapsed %s total"
                                    % fmt_secs(now - started)),
                       "\n".join(log_lines[-200:]), None)
            time.sleep(3)
        else:
            guard["outcome"] = "OVERALL_TIMEOUT"
            cancel(prompt_id)
        stop_watch.set()
        watcher.join(timeout=5)
        status_str = (record or {}).get("status", {}).get("status_str")
        if guard["outcome"] is not None:
            status = "CANCELLED_" + guard["outcome"]
        elif record is None:
            status = "NO_HISTORY"
        else:
            status = "SUCCESS" if status_str == "success" else "FAILED"
        full_log = "\n".join(log_lines)
        final_progress = latest_progress(full_log)
        if final_progress and final_progress.get("observed_s_per_step"):
            rows.append(("s/step", "%.1f (last completed step)"
                         % final_progress["observed_s_per_step"]))
        if guard["projected"]:
            rows.append(("projected denoise", "%.1f min for %d steps"
                         % (guard["projected"] / 60, cfg["steps"])))
        rows += [
            ("status", status + (" / %s" % status_str if status_str else "")),
            ("elapsed", "%.0f s" % (time.perf_counter() - started)),
            ("peak VRAM", "%.0f MiB" % maxima["vram_mib"]),
            ("peak ComfyUI RSS", "%.2f GiB" % maxima["rss_gib"]),
            ("min host free", "%.2f GiB" % maxima["host_low_gib"]),
        ]
        if guard["outcome"]:
            warnings.append("Guard stopped the run: %s" % guard["outcome"])
        saved = {}
        for line in reversed(log_lines):
            if "H3_SAVE_LATENT " in line:
                try:
                    saved = json.loads(line.split("H3_SAVE_LATENT ", 1)[1])
                except Exception:
                    pass
                break
        if saved:
            rows.append(("latent", "%s (%s)" % (saved.get("path"),
                                                gb(saved.get("bytes", 0)))))
        summary_file = OUTPUT_DIR / (cfg["video_prefix"] + ".latent_summary.json")
        if summary_file.is_file():
            summary = json.loads(summary_file.read_text(encoding="utf-8"))
            finite = summary.get("finite")
            rows.append(("latent finite", str(finite)))
            if finite is not None and not all(finite):
                warnings.append("Sampled latent is NOT finite - this run is a NaN result. "
                                "On newer ComfyUI also try Extra flags="
                                "--disable-dynamic-vram (known fp32 NaN trigger).")
            rows.append(("latent shapes", "match" if summary.get("shapes")
                         == cfg["expected_shapes"] else "MISMATCH %s"
                         % summary.get("shapes")))
        if cfg["loras"]:
            rows.append(("loras", ", ".join(
                "%s@%g" % (e["name"], e["strength"])
                for e in cfg["loras"])[:150]))
            turbo = [l for l in log_lines if "[MiniMaxH3TurboLoRA]" in l]
            rows.append(("lora log", turbo[-1][:120] if turbo
                         else "NO LOG - lora may not have applied"))
        if cfg["use_sage"]:
            sage = [l for l in log_lines if "sage attention override active" in l]
            rows.append(("sage", sage[-1][:120] if sage else "NO LOG"))
            if not sage:
                warnings.append("Sage override never logged; attention ran the default "
                                "path.")
        if cfg["text_branch"]:
            tb = [l for l in log_lines if "H3_FP32_TEXT_BRANCH" in l]
            rows.append(("fp32 text branch", tb[-1][:120] if tb else "NO LOG"))
        if cfg["dtype_mode"] == "safe_fp16" \
                and re.search(r"manual cast: torch\.float32", full_log):
            warnings.append("Log still says 'manual cast: torch.float32' - the fp16 fix "
                            "did NOT take effect; check the Setup tab.")
        if "Unexpected architecture type" in full_log:
            warnings.append("GGUF architecture patch did not take - rerun cell 01.")
        import_failures = [l.strip() for l in log_lines
                           if "IMPORT FAILED" in l or "failed to import" in l.lower()]
        if import_failures:
            warnings.append("Import failures: " + "; ".join(import_failures[:3]))
        if re.search(r"CUDA out of memory|torch\.OutOfMemoryError", full_log, re.I):
            hint = ""
            if cfg["dit_choice"] == "abiray_Q4_K_M" and cfg["dtype_mode"] != "bf16" \
                    and cfg["width"] * cfg["height"] >= 864 * 480:
                hint = (" Q4_K_M + fp16 compute exceeds the T4's ~14.5 GiB at this "
                        "geometry; switch to abiray_Q3_K_M or a smaller resolution "
                        "(re-encode is not needed).")
            warnings.append("CUDA OOM seen in the log." + hint)
        H3["last_latent_key"] = cfg["latent_key"]
        metrics = {"status": status, "guard": guard,
                   "s_per_step": final_progress.get("observed_s_per_step")
                   if final_progress else None,
                   "maxima": maxima, "saved_latent": saved, "label": cfg["label"],
                   "steps": cfg["steps"], "dtype_mode": cfg["dtype_mode"],
                   "speed_method": cfg["speed_method"], "sage": cfg["use_sage"]}
        (WORK / ("metrics_%s.json" % cfg["label"])).write_text(
            json.dumps(metrics, indent=2), encoding="utf-8")
    except Exception as exc:
        errors.append("%s: %s" % (type(exc).__name__, str(exc)[:400]))
    finally:
        if stop_watch is not None:
            stop_watch.set()
        sample_log_text = "\n".join(log_lines) if log_lines else tail_comfy_log(
            max(0, COMFY_LOG.stat().st_size - 60000) if COMFY_LOG.exists() else 0, 80)
        (WORK / ("sample_log_%s.txt" % cfg["label"])).write_text(sample_log_text,
                                                                 encoding="utf-8")
        if not keep_server:
            stop_comfy()
    if errors:
        H3["studio_last_ok"] = False
        yield (_status("Sample FAILED", rows, kind="err", note="<br>".join(errors),
                       details=sample_log_text[-5000:]),
               sample_log_text[-5000:], None)
    else:
        status_lookup = next((v for k, v in rows if k == "status"), "")
        H3["studio_last_ok"] = status_lookup.startswith("SUCCESS")
        kind = "ok" if H3["studio_last_ok"] else (
            "warn" if status_lookup.startswith("CANCELLED") else "err")
        yield (_status("Sample " + status_lookup.split(" / ")[0], rows, kind=kind,
                       note="<br>".join(warnings) if warnings else "Decode next.",
                       details=sample_log_text[-5000:] if warnings else ""),
               sample_log_text[-5000:], None)


def op_decode(cfg, keep_server=False):
    latent_path = WORK / "latents" / (cfg["latent_key"] + ".pt")
    summary_file = OUTPUT_DIR / (cfg["video_prefix"] + ".latent_summary.json")
    finite = None
    if summary_file.is_file():
        try:
            finite = json.loads(summary_file.read_text(encoding="utf-8")).get("finite")
        except Exception:
            pass
    if not latent_path.is_file():
        H3["studio_last_ok"] = False
        yield (_status("Decode idle", [("latent", "missing: %s" % latent_path),
                                       ("fix", "run Sample first")], kind="idle"),
               "", None)
        return
    if finite is not None and not all(finite):
        H3["studio_last_ok"] = False
        yield (_status("Decode refused", [("latent", str(latent_path)),
                                          ("finite", str(finite))], kind="warn",
                       note="The latent is non-finite (NaN run); decoding would produce "
                            "noise. Rerun Sample with a working dtype mode."),
               "", None)
        return
    rows, errors, log_lines = [], [], []
    video_file = None
    inspection = {}
    started = time.perf_counter()
    try:
        stop_comfy()
        start_comfy(None)
        H3["server_family"] = "base"
        info = get_object_info()
        H3["object_info"] = info

        def decode_fill(node_type, known):
            spec_in = info[node_type].get("input", {})
            required = spec_in.get("required") or {}
            optional = spec_in.get("optional") or {}
            inputs, unresolved = {}, []
            for name, value in known.items():
                if name in optional and name not in required:
                    inputs[name] = value
            unknown = sorted(set(known) - set(required) - set(optional))
            if unknown:
                raise RuntimeError("%s does not accept %s" % (node_type, unknown))
            for name, spec in required.items():
                if name in known:
                    inputs[name] = known[name]
                    continue
                options = spec[0] if spec else None
                config = spec[1] if len(spec) > 1 and isinstance(spec[1], dict) else {}
                dynamic = config.get("options") \
                    if options == "COMFY_DYNAMICCOMBO_V3" else None
                if dynamic:
                    def needs_nothing(option):
                        return not ((option.get("inputs") or {}).get("required") or {})

                    choice = next((o for o in dynamic if o.get("key") == "auto"
                                   and needs_nothing(o)),
                                  next((o for o in dynamic if needs_nothing(o)), None))
                    if choice is None:
                        unresolved.append("%s: dynamic combo has no free option" % name)
                    else:
                        inputs[name] = choice["key"]
                elif isinstance(options, list) and options:
                    inputs[name] = config.get("default", options[0])
                elif "default" in config:
                    inputs[name] = config["default"]
                else:
                    unresolved.append("%s: %s" % (name, spec))
            if unresolved:
                raise RuntimeError("%s requires inputs with no value here: %s"
                                   % (node_type, "; ".join(unresolved)))
            return inputs

        graph = {
            "1": {"class_type": "VAELoader", "inputs": decode_fill(
                "VAELoader", {"vae_name": cfg["video_vae"]["name"]})},
            "2": {"class_type": "VAELoader", "inputs": decode_fill(
                "VAELoader", {"vae_name": cfg["audio_vae"]["name"]})},
            "3": {"class_type": "H3LoadLatent", "inputs": decode_fill(
                "H3LoadLatent", {"store_dir": str(WORK / "latents"),
                                 "key": cfg["latent_key"]})},
            "4": {"class_type": "VAEDecode", "inputs": decode_fill(
                "VAEDecode", {"samples": ["3", 0], "vae": ["1", 0]})},
            "5": {"class_type": "VAEDecodeAudio", "inputs": decode_fill(
                "VAEDecodeAudio", {"samples": ["3", 0], "vae": ["2", 0]})},
            "6": {"class_type": "CreateVideo", "inputs": decode_fill(
                "CreateVideo", {"images": ["4", 0], "audio": ["5", 0],
                                "fps": float(cfg["fps"])})},
            "7": {"class_type": "SaveVideo", "inputs": decode_fill(
                "SaveVideo", {"video": ["6", 0], "filename_prefix": cfg["video_prefix"]})},
        }
        rows.append(("decode graph", ", ".join("%s:%s" % (i, s["class_type"])
                                               for i, s in sorted(graph.items()))))
        yield (_status("Decode", rows, kind="run",
                       note="starting ComfyUI without the dtype flag (VAE-only RAM) ..."),
               "", None)
        prompt_id = str(uuid.uuid4())
        log_offset = COMFY_LOG.stat().st_size if COMFY_LOG.exists() else 0
        response = requests.post(BASE_URL + "/prompt", timeout=120, json={
            "prompt": graph, "client_id": str(uuid.uuid4()), "prompt_id": prompt_id})
        response.raise_for_status()
        if response.json().get("node_errors"):
            raise RuntimeError("Node errors: %s" % response.json()["node_errors"])
        record, shown = None, log_offset
        while time.perf_counter() - started < 1800:
            proc = H3.get("proc")
            if proc is not None and proc.poll() is not None:
                raise RuntimeError("ComfyUI exited during decode (OOM kill?)")
            new = tail_comfy_log(shown, 400)
            shown = COMFY_LOG.stat().st_size if COMFY_LOG.exists() else shown
            if new:
                log_lines.extend(new.splitlines())
            tail_line = ([l for l in new.splitlines() if l.strip()][-1][:150]
                         .replace("<", "&lt;") if new and new.strip()
                         else (log_lines[-1][:150].replace("<", "&lt;")
                               if log_lines else "(initializing)"))
            yield (_status("Decode", rows, kind="run",
                           note="decoding · elapsed %s<br>%s"
                                % (fmt_secs(time.perf_counter() - started),
                                   tail_line)),
                   "\n".join(log_lines[-200:]), None)
            try:
                history = requests.get(BASE_URL + "/history/" + prompt_id,
                                       timeout=60).json()
                if prompt_id in history:
                    status = history[prompt_id].get("status", {})
                    if status.get("completed") is True \
                            or status.get("status_str") == "error":
                        record = history[prompt_id]
                        break
            except Exception:
                pass
            time.sleep(3)
        if record is None:
            raise TimeoutError("decode did not finish within 1800s")
        if record.get("status", {}).get("status_str") != "success":
            raise RuntimeError("decode failed: %s" % record.get("status"))
        rows.append(("status", "decoded in %.0fs" % (time.perf_counter() - started)))
        produced = []
        for node_output in (record.get("outputs") or {}).values():
            for entry in (node_output.get("images", []) + node_output.get("videos", [])
                          + node_output.get("gifs", [])):
                if isinstance(entry, dict) and entry.get("filename"):
                    produced.append(OUTPUT_DIR / entry.get("subfolder", "")
                                    / entry["filename"])
        if not produced:
            directory = OUTPUT_DIR / "h3_lab"
            produced = sorted([p for p in directory.glob(cfg["label"] + "*")
                               if p.suffix in (".mp4", ".webm")],
                              key=lambda p: p.stat().st_mtime) \
                if directory.is_dir() else []
        produced = [p for p in produced if p.is_file()]
        if not produced:
            raise RuntimeError("SaveVideo wrote nothing findable in %s" % OUTPUT_DIR)
        video_file = produced[-1]
        rows.append(("file", "%s (%.1f MiB)" % (video_file,
                                                video_file.stat().st_size / 2**20)))
        try:
            import av
            with av.open(str(video_file)) as container:
                streams = {s.type: s for s in container.streams}
                inspection["streams"] = sorted(streams)
                if "video" in streams:
                    vs = streams["video"]
                    inspection["video"] = {
                        "codec": vs.codec_context.name,
                        "size": "%dx%d" % (vs.codec_context.width,
                                           vs.codec_context.height),
                        "duration_s": round(float(vs.duration * vs.time_base), 2)
                        if vs.duration else None}
                if "audio" in streams:
                    aus = streams["audio"]
                    inspection["audio"] = {
                        "codec": aus.codec_context.name,
                        "rate": aus.codec_context.sample_rate,
                        "channels": aus.codec_context.channels}
                brightness = []
                index = 0
                for frame in container.decode(video=0):
                    if index % 8 == 0:
                        plane = frame.to_ndarray(format="gray")
                        brightness.append((float(plane.mean()), float(plane.std())))
                    index += 1
                inspection["frames_decoded"] = index
                if brightness:
                    means = [m for m, _ in brightness]
                    stds = [s for _, s in brightness]
                    inspection["mean_luma"] = round(sum(means) / len(means), 2)
                    inspection["near_black"] = sum(1 for m, s in zip(means, stds)
                                                   if m < 4.0 and s < 2.0)
                    inspection["sampled"] = len(means)
        except Exception as exc:
            inspection["error"] = "%s: %s" % (type(exc).__name__, str(exc)[:200])
        rows.append(("inspection", json.dumps(inspection)[:220]))
    except Exception as exc:
        errors.append("%s: %s" % (type(exc).__name__, str(exc)[:400]))
    finally:
        if not keep_server:
            stop_comfy()
    if errors:
        H3["studio_last_ok"] = False
        yield (_status("Decode FAILED", rows, kind="err", note="<br>".join(errors),
                       details="\n".join(log_lines)[-4000:]),
               "\n".join(log_lines[-200:]), None)
    else:
        H3["studio_last_ok"] = True
        note = "Saved under ComfyUI/output and served by the web UI."
        if inspection.get("near_black") and inspection.get("sampled") \
                and inspection["near_black"] == inspection["sampled"]:
            note += " Every sampled frame is dark and flat - the file may be black."
        yield (_status("Decode done", rows, kind="ok", note=note),
               "\n".join(log_lines[-200:]), str(video_file))


def run_h3_job(payload, update, cancel_event):
    cfg = _payload_to_cfg(payload)
    H3["cfg"] = cfg
    H3["studio_last_ok"] = None
    update(label=cfg["label"], dims="%dx%d" % (cfg["width"], cfg["height"]),
           frames=cfg["length"], steps=cfg["steps"], seed=cfg["seed"],
           model="h3", warnings=list(cfg["warnings"]))

    def drive(gen, stage, lo, hi):
        last = None
        update(stage=stage, progress=lo)
        for _status_html, log_text, extra in gen:
            if cancel_event.is_set():
                try:
                    requests.post(BASE_URL + "/interrupt", timeout=15)
                except Exception:
                    pass
                raise RuntimeError("cancelled by user")
            progress = None
            found = latest_progress(log_text or "")
            if found:
                progress = lo + (hi - lo) * found["percent"] / 100.0
            update(stage=stage, progress=progress, log_tail=(log_text or "")[-4000:])
            if extra:
                last = extra
        update(stage=stage, progress=hi)
        return last

    try:
        drive(op_encode(cfg, keep_server=True), "encode", 0.0, 0.08)
        if not H3.get("studio_last_ok"):
            raise RuntimeError("encode failed — check the Console tab")
        drive(op_sample(cfg, keep_server=True), "sampling", 0.08, 0.9)
        if not H3.get("studio_last_ok"):
            raise RuntimeError(H3.get("sample_error")
                               or "sampling failed — check the Console tab")
        path = drive(op_decode(cfg, keep_server=True), "decode", 0.9, 1.0)
        if not H3.get("studio_last_ok"):
            raise RuntimeError("decode failed — check the Console tab")
    except Exception:
        try:
            stop_comfy()
        except Exception:
            pass
        raise
    video_rel = None
    if path:
        try:
            video_rel = Path(str(path)).relative_to(OUTPUT_DIR).as_posix()
        except Exception:
            video_rel = None
    return {
        "video": ("/h3/output?file=%s" % quote(video_rel)) if video_rel else None,
        "meta": {"dims": "%dx%d" % (cfg["width"], cfg["height"]),
                 "frames": cfg["length"]},
        "warnings": list(cfg["warnings"]),
    }


LTX_SIGMAS_BASE = [1.0, 0.99375, 0.9875, 0.98125, 0.975, 0.909375, 0.725,
                   0.421875, 0.0]
LTX_SIGMAS_REFINE = [0.85, 0.725, 0.4219, 0.0]


def _sigmas_text(values, steps):
    """Resample the official distilled schedule when the user changes step count."""
    steps = max(1, int(steps))
    if steps == len(values) - 1:
        return ", ".join("%.6g" % value for value in values)
    out = []
    span = len(values) - 1
    for index in range(steps + 1):
        pos = (index / steps) * span
        low = min(span - 1, int(pos))
        frac = pos - low
        out.append(max(0.0, values[low] * (1.0 - frac) + values[low + 1] * frac))
    out[-1] = 0.0
    return ", ".join("%.6g" % value for value in out)


def build_ltx_cfg(data):
    prompt = str(data.get("prompt") or "").strip()
    if not prompt:
        raise ValueError("prompt is empty")
    mode = str(data.get("mode") or "text").lower()
    if mode not in ("text", "image", "firstlast"):
        raise ValueError("unknown LTX mode %r" % mode)
    width = int(data.get("width") or 640)
    height = int(data.get("height") or 384)
    if width < 256 or height < 256 or width % 64 or height % 64:
        raise ValueError("LTX width/height must be >=256 and divisible by 64 "
                         "(the half-resolution base pass needs multiples of 32); "
                         "got %dx%d" % (width, height))
    frames = int(data.get("frames") or 121)
    frames = max(9, min(361, ((frames - 1) // 8) * 8 + 1))
    _vram_preflight("LTX-2.5", LTX_RESIDENT_GIB, width, height, frames,
                    "Lower the resolution (Preview 640x384 is the tested T4 default), "
                    "shorten the clip, or use a runtime with more VRAM.")
    fps = int(data.get("fps") or 24)
    if fps < 8 or fps > 30:
        raise ValueError("fps must be between 8 and 30")
    steps_base = max(2, min(20, int(data.get("steps_base") or 8)))
    steps_refine = max(1, min(12, int(data.get("steps_refine") or 3)))
    sampler = str(data.get("sampler") or "euler_ancestral")
    if sampler not in ("euler_ancestral", "euler", "res_multistep", "dpmpp_2m"):
        sampler = "euler_ancestral"
    video_cfg = max(0.0, min(5.0, float(data.get("video_cfg") or 1.0)))
    audio_cfg = max(0.0, min(5.0, float(data.get("audio_cfg") or 1.0)))
    negative = str(data.get("negative") or "").strip()
    first_frame = str(data.get("first_frame") or "").strip() \
        if mode in ("image", "firstlast") else ""
    last_frame = str(data.get("last_frame") or "").strip() \
        if mode == "firstlast" else ""
    if mode == "image" and not first_frame:
        raise ValueError("LTX Image mode needs a first frame (upload it in Frames)")
    if mode == "firstlast" and not (first_frame and last_frame):
        raise ValueError("LTX First+Last needs both a first and a last frame")
    loras_cfg = []
    for entry in _clean_loras(data.get("loras")):
        lora_path = LORA_DIR / entry["name"]
        if not lora_path.is_file():
            raise ValueError("LTX LoRA not found: %s (download it in the web UI)"
                             % lora_path)
        loras_cfg.append({"name": entry["name"], "strength": float(entry["strength"])})
    warnings = []
    if width * height > 768 * 448:
        warnings.append("LTX at %dx%d is heavy for a free T4; the 22B DiT may OOM or "
                        "take a long time. 640x384 is the tested default."
                        % (width, height))
    if steps_base != 8:
        warnings.append("The distilled LTX-2.5 schedule is tuned for 8 base steps; "
                        "%d steps resamples the same curve and may reduce quality."
                        % steps_base)
    if mode != "text":
        warnings.append("LTX Image/First+Last uses the official single-stage guide graph "
                        "at full resolution (no x2 refine pass).")
    if fps != 24:
        warnings.append("LTX-2.5 is trained around 24-25 fps; %d fps may soften audio "
                        "sync." % fps)

    first_sig, last_sig = _frame_sig(first_frame), _frame_sig(last_frame)
    text_key = "t" + hashlib.sha1(json.dumps({
        "prompt": prompt, "negative": negative, "fps": fps,
    }, sort_keys=True).encode("utf-8")).hexdigest()[:16]
    run_key = "v" + hashlib.sha1(json.dumps({
        "text": text_key, "seed": int(data.get("seed") or 0), "mode": mode,
        "width": width, "height": height, "frames": frames, "fps": fps,
        "steps_base": steps_base, "steps_refine": steps_refine,
        "sigmas_base": _sigmas_text(LTX_SIGMAS_BASE, steps_base),
        "sigmas_refine": _sigmas_text(LTX_SIGMAS_REFINE, steps_refine),
        "sampler": sampler, "video_cfg": video_cfg, "audio_cfg": audio_cfg,
        "loras": [[entry["name"], entry["strength"]] for entry in loras_cfg],
        "first": first_sig, "last": last_sig, "dit": LTX_FILES["dit"]["name"],
        "upscaler": LTX_FILES["upscaler"]["name"],
    }, sort_keys=True).encode("utf-8")).hexdigest()[:16]
    tag = {"text": "t2v", "image": "i2v", "firstlast": "flf"}[mode]
    # The run key (prompt, seed, geometry, sigmas, LoRAs) is part of the name so a
    # changed LoRA stack never reuses an older clip from the same geometry.
    label = "ltx_%s_%dx%d_%df_%d%+dr_%s" % (
        tag, width, height, frames, steps_base, steps_refine, run_key[:8])
    cfg = {
        "mode": mode, "prompt": prompt, "negative": negative,
        "width": width, "height": height, "frames": frames, "fps": fps,
        "seed": int(data.get("seed") or 0),
        "steps_base": steps_base, "steps_refine": steps_refine,
        "sampler": sampler, "video_cfg": video_cfg, "audio_cfg": audio_cfg,
        "sigmas_base": _sigmas_text(LTX_SIGMAS_BASE, steps_base),
        "sigmas_refine": _sigmas_text(LTX_SIGMAS_REFINE, steps_refine),
        "base_width": width // 2, "base_height": height // 2,
        "first_frame": first_frame, "last_frame": last_frame,
        "loras": loras_cfg,
        "text_key": text_key, "key": run_key, "label": label,
        "video_prefix": "ltx25lab/" + label,
        "cond_pos": str(COND_DIR / (text_key + "_pos.pt")),
        "cond_neg": str(COND_DIR / (text_key + "_neg.pt")),
        "base_video": str(LAT_DIR / (run_key + "_basevideo.pt")),
        "base_audio": str(LAT_DIR / (run_key + "_baseaudio.pt")),
        "up_video": str(LAT_DIR / (run_key + "_upvideo.pt")),
        "video": str(LAT_DIR / (run_key + "_video.pt")),
        "audio": str(LAT_DIR / (run_key + "_audio.pt")),
        "dit": LTX_FILES["dit"]["name"], "te": LTX_FILES["te"]["name"],
        "vae_video": LTX_FILES["vae_video"]["name"],
        "vae_audio": LTX_FILES["vae_audio"]["name"],
        "upscaler": LTX_FILES["upscaler"]["name"],
        "budget_s": int(float(data.get("budget") or 20) * 60),
        "warnings": warnings,
    }
    return cfg


def _ltx_require_nodes(info, cfg):
    common = {"CLIPLoaderGGUF", "CLIPTextEncode", "LTXVConditioning", "UnetLoaderGGUF",
              "LTXVDualCFGGuider", "RandomNoise", "KSamplerSelect", "ManualSigmas",
              "EmptyLTXVLatentVideo", "VAELoader", "LTXVEmptyLatentAudio",
              "LTXVConcatAVLatent", "SamplerCustomAdvanced", "LTXVSeparateAVLatent",
              "VAEDecodeTiled", "LTXVAudioVAEDecode", "CreateVideo", "SaveVideo",
              "LTX25SaveConditioning", "LTX25LoadConditioning", "LTX25SaveLatent",
              "LTX25LoadLatent"}
    if cfg["mode"] == "text":
        common |= {"LatentUpscaleModelLoader", "LTXVLatentUpsampler"}
    else:
        common |= {"LTXVAddGuide", "LTXVCropGuides", "LTXVPreprocess", "LoadImage"}
    if cfg.get("loras"):
        common |= {"LoraLoaderModelOnly"}
    missing = sorted(node for node in common if node not in info)
    if missing:
        raise RuntimeError("LTX nodes not loaded: %s - rerun cell 01"
                           % ", ".join(missing))
    types = info["CLIPLoaderGGUF"]["input"]["required"]["type"][0]
    if "ltxv" not in types:
        raise RuntimeError("CLIPLoaderGGUF type list lacks 'ltxv': %s" % types[:20])


def _ltx_find_video(cfg):
    directory = OUTPUT_DIR / "ltx25lab"
    if not directory.is_dir():
        return None
    files = sorted([path for path in directory.glob(cfg["label"] + "*")
                    if path.suffix in (".mp4", ".webm")],
                   key=lambda path: path.stat().st_mtime)
    return files[-1] if files else None


def _ltx_video_from_record(record):
    produced = []
    for node_output in (record.get("outputs") or {}).values():
        entries = (node_output.get("images") or []) + (node_output.get("videos") or []) \
            + (node_output.get("gifs") or [])
        for entry in entries:
            if isinstance(entry, dict) and entry.get("filename"):
                produced.append(OUTPUT_DIR / entry.get("subfolder", "")
                                / entry["filename"])
    return [path for path in produced if path.is_file()]


def _ltx_graph_run(graph, title, cfg, timeout=3600):
    log_offset = COMFY_LOG.stat().st_size if COMFY_LOG.exists() else 0
    started = time.perf_counter()
    log_lines = []
    rows = [("label", cfg["label"]), ("mode", cfg["mode"])]
    prompt_id = str(uuid.uuid4())
    response = requests.post(BASE_URL + "/prompt", timeout=120, json={
        "prompt": graph, "client_id": str(uuid.uuid4()), "prompt_id": prompt_id})
    if response.status_code != 200:
        raise RuntimeError("%s: /prompt HTTP %d: %s"
                           % (title, response.status_code, response.text[-1000:]))
    body = response.json()
    if body.get("node_errors"):
        raise RuntimeError("Node errors: %s" % body["node_errors"])
    shown = log_offset
    yield (_status(title, rows, kind="run", note="submitted to ComfyUI ..."), "", None)
    record = None
    while time.perf_counter() - started < timeout:
        proc = H3.get("proc")
        if proc is not None and proc.poll() is not None:
            raise RuntimeError("ComfyUI exited during %s (OOM kill?)" % title)
        new = tail_comfy_log(shown, 500)
        shown = COMFY_LOG.stat().st_size if COMFY_LOG.exists() else shown
        if new:
            log_lines.extend(new.splitlines())
        text = "\n".join(log_lines[-400:])
        progress = latest_progress(text)
        if progress:
            live_rows = rows + [
                ("sampling", "%s  step %d/%d" % (
                    pbar(progress["steps_completed"] / max(1, progress["steps_total"])),
                    progress["steps_completed"], progress["steps_total"])),
                ("s/step", "%.1f" % progress["observed_s_per_step"]
                 if progress.get("observed_s_per_step") else "measuring ..."),
                ("elapsed", fmt_secs(time.perf_counter() - started)),
            ]
            note = "working · elapsed %s" % fmt_secs(time.perf_counter() - started)
        else:
            lines = [line for line in new.splitlines() if line.strip()]
            live_rows = rows + [("elapsed", fmt_secs(time.perf_counter() - started))]
            note = (lines[-1][:180].replace("<", "&lt;") if lines
                    else "loading model / waiting for the sampler")
        yield (_status(title, live_rows, kind="run", note=note),
               "\n".join(log_lines[-250:]), None)
        try:
            history = requests.get(BASE_URL + "/history/" + prompt_id,
                                   timeout=60).json()
            if prompt_id in history:
                status = history[prompt_id].get("status", {})
                if status.get("completed") is True \
                        or status.get("status_str") == "error":
                    record = history[prompt_id]
                    break
        except Exception:
            pass
        time.sleep(2)
    if record is None:
        raise TimeoutError("%s did not finish within %ds" % (title, timeout))
    status_str = record.get("status", {}).get("status_str")
    if status_str != "success":
        raise RuntimeError("%s failed: %s" % (title, json.dumps(
            record.get("status", {}).get("messages", []))[-1200:]))
    LTX["last_record"] = record
    rows.append(("elapsed", "%.0fs" % (time.perf_counter() - started)))
    yield (_status(title, rows, kind="ok", note="stage complete"),
           "\n".join(log_lines[-250:]), None)


def _ltx_encode_graph(cfg):
    return {
        "1": {"class_type": "CLIPLoaderGGUF",
              "inputs": {"clip_name": cfg["te"], "type": "ltxv"}},
        "2": {"class_type": "CLIPTextEncode",
              "inputs": {"clip": ["1", 0], "text": cfg["prompt"]}},
        "3": {"class_type": "CLIPTextEncode",
              "inputs": {"clip": ["1", 0], "text": cfg["negative"]}},
        "4": {"class_type": "LTXVConditioning",
              "inputs": {"positive": ["2", 0], "negative": ["3", 0],
                         "frame_rate": cfg["fps"]}},
        "5": {"class_type": "LTX25SaveConditioning",
              "inputs": {"conditioning": ["4", 0], "store_dir": str(COND_DIR),
                         "key": cfg["text_key"] + "_pos"}},
        "6": {"class_type": "LTX25SaveConditioning",
              "inputs": {"conditioning": ["4", 1], "store_dir": str(COND_DIR),
                         "key": cfg["text_key"] + "_neg"}},
    }


def _ltx_apply_loras(graph, model_node, guider_id, cfg):
    """Chain one LoraLoaderModelOnly per entry after the GGUF DiT loader."""
    source = [model_node, 0]
    for index, entry in enumerate(cfg.get("loras") or []):
        node_id = "lora%d" % index
        graph[node_id] = {"class_type": "LoraLoaderModelOnly", "inputs": {
            "model": source, "lora_name": entry["name"],
            "strength_model": entry["strength"]}}
        source = [node_id, 0]
    graph[guider_id]["inputs"]["model"] = source
    return graph


def _ltx_sample_base_graph(cfg):
    graph = {
        "1": {"class_type": "UnetLoaderGGUF",
              "inputs": {"unet_name": cfg["dit"]}},
        "2": {"class_type": "LTX25LoadConditioning",
              "inputs": {"store_dir": str(COND_DIR),
                         "key": cfg["text_key"] + "_pos"}},
        "3": {"class_type": "LTX25LoadConditioning",
              "inputs": {"store_dir": str(COND_DIR),
                         "key": cfg["text_key"] + "_neg"}},
        "4": {"class_type": "LTXVDualCFGGuider",
              "inputs": {"model": ["1", 0], "positive": ["2", 0], "negative": ["3", 0],
                         "video_cfg": cfg["video_cfg"], "audio_cfg": cfg["audio_cfg"]}},
        "5": {"class_type": "RandomNoise", "inputs": {"noise_seed": cfg["seed"]}},
        "6": {"class_type": "KSamplerSelect",
              "inputs": {"sampler_name": cfg["sampler"]}},
        "7": {"class_type": "ManualSigmas",
              "inputs": {"sigmas": cfg["sigmas_base"]}},
        "8": {"class_type": "EmptyLTXVLatentVideo",
              "inputs": {"width": cfg["base_width"], "height": cfg["base_height"],
                         "length": cfg["frames"], "batch_size": 1}},
        "9": {"class_type": "VAELoader", "inputs": {"vae_name": cfg["vae_audio"]}},
        "10": {"class_type": "LTXVEmptyLatentAudio",
               "inputs": {"frames_number": cfg["frames"],
                          "frame_rate": cfg["fps"], "batch_size": 1,
                          "audio_vae": ["9", 0]}},
        "11": {"class_type": "LTXVConcatAVLatent",
               "inputs": {"video_latent": ["8", 0], "audio_latent": ["10", 0]}},
        "12": {"class_type": "SamplerCustomAdvanced",
               "inputs": {"noise": ["5", 0], "guider": ["4", 0], "sampler": ["6", 0],
                          "sigmas": ["7", 0], "latent_image": ["11", 0]}},
        "13": {"class_type": "LTXVSeparateAVLatent",
               "inputs": {"av_latent": ["12", 0]}},
        "14": {"class_type": "LTX25SaveLatent",
               "inputs": {"samples": ["13", 0], "store_dir": str(LAT_DIR),
                          "key": cfg["key"] + "_basevideo"}},
        "15": {"class_type": "LTX25SaveLatent",
               "inputs": {"samples": ["13", 1], "store_dir": str(LAT_DIR),
                          "key": cfg["key"] + "_baseaudio"}},
    }
    return _ltx_apply_loras(graph, "1", "4", cfg)


def _ltx_upscale_graph(cfg):
    return {
        "1": {"class_type": "LTX25LoadLatent",
              "inputs": {"store_dir": str(LAT_DIR),
                         "key": cfg["key"] + "_basevideo"}},
        "2": {"class_type": "LatentUpscaleModelLoader",
              "inputs": {"model_name": cfg["upscaler"]}},
        "3": {"class_type": "VAELoader", "inputs": {"vae_name": cfg["vae_video"]}},
        "4": {"class_type": "LTXVLatentUpsampler",
              "inputs": {"samples": ["1", 0], "upscale_model": ["2", 0],
                         "vae": ["3", 0]}},
        "5": {"class_type": "LTX25SaveLatent",
              "inputs": {"samples": ["4", 0], "store_dir": str(LAT_DIR),
                         "key": cfg["key"] + "_upvideo"}},
    }


def _ltx_refine_graph(cfg):
    return {
        "1": {"class_type": "UnetLoaderGGUF",
              "inputs": {"unet_name": cfg["dit"]}},
        "2": {"class_type": "LTX25LoadConditioning",
              "inputs": {"store_dir": str(COND_DIR),
                         "key": cfg["text_key"] + "_pos"}},
        "3": {"class_type": "LTX25LoadConditioning",
              "inputs": {"store_dir": str(COND_DIR),
                         "key": cfg["text_key"] + "_neg"}},
        "4": {"class_type": "LTXVDualCFGGuider",
              "inputs": {"model": ["1", 0], "positive": ["2", 0], "negative": ["3", 0],
                         "video_cfg": cfg["video_cfg"], "audio_cfg": cfg["audio_cfg"]}},
        "5": {"class_type": "RandomNoise", "inputs": {"noise_seed": cfg["seed"]}},
        "6": {"class_type": "KSamplerSelect",
              "inputs": {"sampler_name": cfg["sampler"]}},
        "7": {"class_type": "ManualSigmas",
              "inputs": {"sigmas": cfg["sigmas_refine"]}},
        "8": {"class_type": "LTX25LoadLatent",
              "inputs": {"store_dir": str(LAT_DIR),
                         "key": cfg["key"] + "_upvideo"}},
        "9": {"class_type": "LTX25LoadLatent",
              "inputs": {"store_dir": str(LAT_DIR),
                         "key": cfg["key"] + "_baseaudio"}},
        "10": {"class_type": "LTXVConcatAVLatent",
               "inputs": {"video_latent": ["8", 0], "audio_latent": ["9", 0]}},
        "11": {"class_type": "SamplerCustomAdvanced",
               "inputs": {"noise": ["5", 0], "guider": ["4", 0], "sampler": ["6", 0],
                          "sigmas": ["7", 0], "latent_image": ["10", 0]}},
        "12": {"class_type": "LTXVSeparateAVLatent",
               "inputs": {"av_latent": ["11", 0]}},
        "13": {"class_type": "LTX25SaveLatent",
               "inputs": {"samples": ["12", 0], "store_dir": str(LAT_DIR),
                          "key": cfg["key"] + "_video"}},
        "14": {"class_type": "LTX25SaveLatent",
               "inputs": {"samples": ["12", 1], "store_dir": str(LAT_DIR),
                          "key": cfg["key"] + "_audio"}},
    }
    return _ltx_apply_loras(graph, "1", "4", cfg)


def _ltx_guide_graph(cfg):
    first_last = cfg["mode"] == "firstlast"
    last_guide = "25" if first_last else "10"
    graph = {
        "1": {"class_type": "CLIPLoaderGGUF",
              "inputs": {"clip_name": cfg["te"], "type": "ltxv"}},
        "2": {"class_type": "CLIPTextEncode",
              "inputs": {"clip": ["1", 0], "text": cfg["prompt"]}},
        "3": {"class_type": "CLIPTextEncode",
              "inputs": {"clip": ["1", 0], "text": cfg["negative"]}},
        "4": {"class_type": "LTXVConditioning",
              "inputs": {"positive": ["2", 0], "negative": ["3", 0],
                         "frame_rate": cfg["fps"]}},
        "5": {"class_type": "UnetLoaderGGUF",
              "inputs": {"unet_name": cfg["dit"]}},
        "6": {"class_type": "VAELoader", "inputs": {"vae_name": cfg["vae_video"]}},
        "7": {"class_type": "EmptyLTXVLatentVideo",
              "inputs": {"width": cfg["width"], "height": cfg["height"],
                         "length": cfg["frames"], "batch_size": 1}},
        "8": {"class_type": "LoadImage", "inputs": {"image": cfg["first_image"]}},
        "9": {"class_type": "LTXVPreprocess",
              "inputs": {"image": ["8", 0], "img_compression": 18}},
        "10": {"class_type": "LTXVAddGuide",
               "inputs": {"positive": ["4", 0], "negative": ["4", 1],
                          "vae": ["6", 0], "latent": ["7", 0], "image": ["9", 0],
                          "frame_idx": 0, "strength": 0.7}},
        "11": {"class_type": "VAELoader", "inputs": {"vae_name": cfg["vae_audio"]}},
        "12": {"class_type": "LTXVEmptyLatentAudio",
               "inputs": {"frames_number": cfg["frames"],
                          "frame_rate": cfg["fps"], "batch_size": 1,
                          "audio_vae": ["11", 0]}},
        "13": {"class_type": "LTXVConcatAVLatent",
               "inputs": {"video_latent": [last_guide, 2], "audio_latent": ["12", 0]}},
        "14": {"class_type": "LTXVDualCFGGuider",
               "inputs": {"model": ["5", 0], "positive": [last_guide, 0],
                          "negative": [last_guide, 1],
                          "video_cfg": cfg["video_cfg"], "audio_cfg": cfg["audio_cfg"]}},
        "15": {"class_type": "RandomNoise", "inputs": {"noise_seed": cfg["seed"]}},
        "16": {"class_type": "KSamplerSelect",
               "inputs": {"sampler_name": cfg["sampler"]}},
        "17": {"class_type": "ManualSigmas",
               "inputs": {"sigmas": cfg["sigmas_base"]}},
        "18": {"class_type": "SamplerCustomAdvanced",
               "inputs": {"noise": ["15", 0], "guider": ["14", 0], "sampler": ["16", 0],
                          "sigmas": ["17", 0], "latent_image": ["13", 0]}},
        "19": {"class_type": "LTXVSeparateAVLatent",
               "inputs": {"av_latent": ["18", 0]}},
        "20": {"class_type": "LTXVCropGuides",
               "inputs": {"positive": [last_guide, 0], "negative": [last_guide, 1],
                          "latent": ["19", 0]}},
        "21": {"class_type": "LTX25SaveLatent",
               "inputs": {"samples": ["20", 2], "store_dir": str(LAT_DIR),
                          "key": cfg["key"] + "_video"}},
        "22": {"class_type": "LTX25SaveLatent",
               "inputs": {"samples": ["19", 1], "store_dir": str(LAT_DIR),
                          "key": cfg["key"] + "_audio"}},
    }
    if first_last:
        graph["23"] = {"class_type": "LoadImage",
                       "inputs": {"image": cfg["last_image"]}}
        graph["24"] = {"class_type": "LTXVPreprocess",
                       "inputs": {"image": ["23", 0], "img_compression": 18}}
        graph["25"] = {"class_type": "LTXVAddGuide",
                       "inputs": {"positive": ["10", 0], "negative": ["10", 1],
                                  "vae": ["6", 0], "latent": ["10", 2],
                                  "image": ["24", 0], "frame_idx": -1,
                                  "strength": 0.7}}
    return _ltx_apply_loras(graph, "5", "14", cfg)


def _ltx_decode_graph(cfg):
    return {
        "1": {"class_type": "LTX25LoadLatent",
              "inputs": {"store_dir": str(LAT_DIR), "key": cfg["key"] + "_video"}},
        "2": {"class_type": "VAELoader", "inputs": {"vae_name": cfg["vae_video"]}},
        "3": {"class_type": "VAEDecodeTiled",
              "inputs": {"samples": ["1", 0], "vae": ["2", 0], "tile_size": 512,
                         "overlap": 64, "temporal_size": 64, "temporal_overlap": 16}},
        "4": {"class_type": "LTX25LoadLatent",
              "inputs": {"store_dir": str(LAT_DIR), "key": cfg["key"] + "_audio"}},
        "5": {"class_type": "VAELoader", "inputs": {"vae_name": cfg["vae_audio"]}},
        "6": {"class_type": "LTXVAudioVAEDecode",
              "inputs": {"samples": ["4", 0], "audio_vae": ["5", 0]}},
        "7": {"class_type": "CreateVideo",
              "inputs": {"images": ["3", 0], "audio": ["6", 0], "fps": cfg["fps"]}},
        "8": {"class_type": "SaveVideo",
              "inputs": {"video": ["7", 0], "filename_prefix": cfg["video_prefix"],
                         "format": "auto", "codec": "auto"}},
    }


def _op_ltx_encode(cfg):
    if all(Path(cfg[key]).is_file() and Path(cfg[key]).stat().st_size > 1000
           for key in ("cond_pos", "cond_neg")):
        yield (_status("LTX encode recalled", [("text key", cfg["text_key"])], kind="ok",
                       note="Text conditioning already cached."), "", None)
        return
    for item in _ltx_graph_run(_ltx_encode_graph(cfg), "LTX encode", cfg, 2400):
        yield item


def _op_ltx_sample_base(cfg):
    if all(Path(cfg[key]).is_file() for key in ("base_video", "base_audio")):
        yield (_status("LTX base recalled", [("run key", cfg["key"])], kind="ok",
                       note="Base latents already cached."), "", None)
        return
    for item in _ltx_graph_run(_ltx_sample_base_graph(cfg), "LTX base sampling", cfg,
                               3600):
        yield item


def _op_ltx_upscale(cfg):
    if Path(cfg["up_video"]).is_file():
        yield (_status("LTX upscale recalled", [("run key", cfg["key"])], kind="ok",
                       note="Upscaled latent already cached."), "", None)
        return
    for item in _ltx_graph_run(_ltx_upscale_graph(cfg), "LTX x2 upscale", cfg, 1200):
        yield item


def _op_ltx_sample_refine(cfg):
    if all(Path(cfg[key]).is_file() for key in ("video", "audio")):
        yield (_status("LTX refine recalled", [("run key", cfg["key"])], kind="ok",
                       note="Refined latents already cached."), "", None)
        return
    for item in _ltx_graph_run(_ltx_refine_graph(cfg), "LTX refine", cfg, 3600):
        yield item


def _op_ltx_sample_guides(cfg):
    if all(Path(cfg[key]).is_file() for key in ("video", "audio")):
        yield (_status("LTX guide recalled", [("run key", cfg["key"])], kind="ok",
                       note="Guided latents already cached."), "", None)
        return
    cfg["first_image"] = prepare_image(cfg["first_frame"], "first_frame")
    if cfg["mode"] == "firstlast":
        cfg["last_image"] = prepare_image(cfg["last_frame"], "last_frame")
    for item in _ltx_graph_run(_ltx_guide_graph(cfg), "LTX guided sampling", cfg, 3600):
        yield item


def _op_ltx_decode(cfg):
    existing = _ltx_find_video(cfg)
    if existing is not None:
        yield (_status("LTX decode recalled", [("file", str(existing))], kind="ok",
                       note="Video already exists; serving it."), "", str(existing))
        return
    for item in _ltx_graph_run(_ltx_decode_graph(cfg), "LTX decode", cfg, 2700):
        yield item
    record = LTX.get("last_record") or {}
    produced = _ltx_video_from_record(record)
    video = produced[-1] if produced else _ltx_find_video(cfg)
    if video is None:
        raise RuntimeError("SaveVideo wrote no mp4 under %s" % (OUTPUT_DIR / "ltx25lab"))
    yield (_status("LTX decode done", [
        ("file", "%s (%.1f MiB)" % (video, video.stat().st_size / 2**20))],
        kind="ok", note="Saved under ComfyUI/output and served by the web UI."),
        "", str(video))


def ensure_ltx_server():
    stats = None
    try:
        stats = requests.get(BASE_URL + "/system_stats", timeout=5).json()
    except Exception:
        stats = None
    if stats is not None and H3.get("server_family") == "ltx":
        # Same stale-log guard as ensure_server: progress is parsed from new lines only.
        offset = COMFY_LOG.stat().st_size if COMFY_LOG.exists() else 0
        return stats, offset
    stop_comfy()
    stats, offset = start_comfy(None, wait_s=1200)
    H3["server_family"] = "ltx"
    return stats, offset


def run_ltx_job(payload, update, cancel_event):
    cfg = build_ltx_cfg(payload)
    LTX["cfg"] = cfg
    update(label=cfg["label"], dims="%dx%d" % (cfg["width"], cfg["height"]),
           frames=cfg["frames"],
           steps=(cfg["steps_base"] + (cfg["steps_refine"]
                                       if cfg["mode"] == "text" else 0)),
           seed=cfg["seed"], model="ltx", warnings=list(cfg["warnings"]))

    def drive(gen, stage, low, high):
        last = None
        update(stage=stage, progress=low)
        for _status_html, log_text, extra in gen:
            if cancel_event.is_set():
                try:
                    requests.post(BASE_URL + "/interrupt", timeout=15)
                except Exception:
                    pass
                raise RuntimeError("cancelled by user")
            found = latest_progress(log_text or "")
            progress = low + (high - low) * found["percent"] / 100.0 if found else low
            update(stage=stage, progress=progress,
                   log_tail=(log_text or "")[-4000:])
            if extra:
                last = extra
        update(stage=stage, progress=high)
        return last

    try:
        ensure_ltx_server()
        info = get_object_info()
        _ltx_require_nodes(info, cfg)
        drive(_op_ltx_encode(cfg), "encode", 0.0, 0.05)
        if cfg["mode"] == "text":
            drive(_op_ltx_sample_base(cfg), "sampling base", 0.05, 0.55)
            drive(_op_ltx_upscale(cfg), "upscaling", 0.55, 0.62)
            drive(_op_ltx_sample_refine(cfg), "refining", 0.62, 0.90)
        else:
            drive(_op_ltx_sample_guides(cfg), "sampling", 0.05, 0.90)
        path = drive(_op_ltx_decode(cfg), "decode", 0.90, 1.0)
        if not path:
            path = _ltx_find_video(cfg)
        if not path:
            raise RuntimeError("LTX produced no video file")
    except Exception:
        try:
            stop_comfy()
        except Exception:
            pass
        raise
    video_rel = Path(str(path)).relative_to(OUTPUT_DIR).as_posix()
    return {
        "video": "/h3/output?file=%s" % quote(video_rel),
        "meta": {"dims": "%dx%d" % (cfg["width"], cfg["height"]),
                 "frames": cfg["frames"],
                 "steps": cfg["steps_base"] + (cfg["steps_refine"]
                                               if cfg["mode"] == "text" else 0)},
        "warnings": list(cfg["warnings"]),
    }


def run_job(payload, update, cancel_event):
    """Route a Studio job to the selected model family."""
    model = str(payload.get("model") or "h3").strip().lower()
    if model in ("ltx", "ltx25", "ltx-2.5", "ltx2.5"):
        return run_ltx_job(payload, update, cancel_event)
    if model in ("h3", "minimax", "minimax_h3"):
        return run_h3_job(payload, update, cancel_event)
    raise ValueError("unknown model %r (expected h3 or ltx)" % model)


def _pick_port(start=STUDIO_PORT):
    for port in range(start, start + 50):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
            probe.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            try:
                probe.bind(("127.0.0.1", port))
                return port
            except OSError:
                continue
    raise RuntimeError("no free port in %d..%d" % (start, start + 49))


def _free_port(port):
    freed = []
    try:
        for conn in psutil.net_connections(kind="inet"):
            try:
                if getattr(conn, "status", None) != psutil.CONN_LISTEN:
                    continue
                if not conn.laddr or conn.laddr.port != port or not conn.pid:
                    continue
                os.kill(conn.pid, signal.SIGTERM)
                freed.append(conn.pid)
            except Exception:
                pass
    except Exception:
        pass
    if freed:
        time.sleep(1.0)
    for pid in list(freed):
        try:
            if psutil.pid_exists(pid):
                os.kill(pid, signal.SIGKILL)
        except Exception:
            pass
    return freed


def _ensure_repo():
    """Refresh the Studio checkout at STUDIO_BRANCH; one-line state for the output box."""
    if (REPO_DIR / ".git").is_dir():
        rc, out = sh(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1",
                      "origin", STUDIO_BRANCH], timeout=600)
        if rc == 0:
            sh(["git", "-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD"], timeout=120)
            repo_state = "updated to origin/%s" % STUDIO_BRANCH
        else:
            tail = out.strip().splitlines()[-1][:140] if out.strip() else "fetch failed"
            repo_state = "update skipped (%s)" % tail
    else:
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        rc, out = sh(["git", "clone", "--depth", "1", "--branch", STUDIO_BRANCH,
                      REPO_URL, str(REPO_DIR)], timeout=900)
        if rc:
            raise RuntimeError("H3-Studio clone of %s failed: %s"
                               % (STUDIO_BRANCH, out[-400:]))
        repo_state = "cloned %s to %s" % (STUDIO_BRANCH, REPO_DIR)
    rc, out = sh(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], timeout=60)
    if rc == 0 and out.strip():
        repo_state += " @ " + out.strip().splitlines()[0]
    if not (REPO_DIR / "server" / "h3_backend.py").is_file():
        raise RuntimeError("H3-Studio checkout has no server/h3_backend.py")
    return repo_state


def _ensure_node():
    """Local Node (Vite 8 needs >= 20.19; Colab's system node is often older)."""
    node_bin = WORK / "node" / "bin"
    node = node_bin / "node"
    if node.is_file():
        rc, out = sh([str(node), "--version"], timeout=60)
        if rc == 0 and out.strip():
            return node_bin, out.strip().splitlines()[0]
    archive = WORK / ("node-%s-linux-x64.tar.xz" % NODE_VERSION)
    if not archive.is_file() or archive.stat().st_size < 10_000_000:
        url = ("https://nodejs.org/dist/%s/node-%s-linux-x64.tar.xz"
               % (NODE_VERSION, NODE_VERSION))
        response = requests.get(url, timeout=600)
        response.raise_for_status()
        archive.write_bytes(response.content)
    shutil.rmtree(WORK / "node", ignore_errors=True)
    shutil.rmtree(WORK / ("node-%s-linux-x64" % NODE_VERSION), ignore_errors=True)
    rc, out = sh(["tar", "-xJf", str(archive), "-C", str(WORK)], timeout=300)
    if rc:
        raise RuntimeError("node extract failed: %s" % out[-300:])
    (WORK / ("node-%s-linux-x64" % NODE_VERSION)).rename(WORK / "node")
    rc, out = sh([str(node), "--version"], timeout=60)
    if rc:
        raise RuntimeError("node bootstrap failed: %s" % out[-300:])
    return node_bin, out.strip().splitlines()[0]


def _build_ui():
    """npm ci + vite build inside the checkout; one-line state for the output box."""
    node_bin, node_version = _ensure_node()
    env = {"PATH": "%s:%s" % (node_bin, os.environ.get("PATH", ""))}
    for command in (["npm", "ci", "--no-audit", "--no-fund"], ["npm", "run", "build"]):
        rc, out = sh(command, cwd=str(REPO_DIR), timeout=900, env=env)
        if rc:
            raise RuntimeError("%s failed: %s" % (" ".join(command), out[-400:]))
    if not (REPO_DIR / "dist" / "index.html").is_file():
        raise RuntimeError("vite build produced no dist/index.html")
    return "built from source (node %s)" % node_version.lstrip("v")


def _ensure_cloudflared():
    if CLOUDFLARED.is_file() and CLOUDFLARED.stat().st_size > 10_000_000:
        return
    url = ("https://github.com/cloudflare/cloudflared/releases/latest/download/"
           "cloudflared-linux-amd64")
    response = requests.get(url, timeout=600)
    response.raise_for_status()
    CLOUDFLARED.write_bytes(response.content)
    CLOUDFLARED.chmod(0o755)


def _start_tunnel(port, timeout_s=90):
    proc = subprocess.Popen(
        [str(CLOUDFLARED), "tunnel", "--url", "http://127.0.0.1:%d" % port,
         "--protocol", "http2", "--no-autoupdate"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        start_new_session=True)
    pattern = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
    found = {}

    def reader():
        try:
            for line in proc.stdout:
                if not found.get("url"):
                    match = pattern.search(line)
                    if match:
                        found["url"] = match.group(0)
                found.setdefault("tail", [])
                found["tail"] = (found["tail"] + [line.strip()])[-8:]
        except Exception:
            pass

    threading.Thread(target=reader, daemon=True, name="cloudflared-reader").start()
    started = time.time()
    while time.time() - started < timeout_s:
        if found.get("url"):
            return proc, found["url"]
        if proc.poll() is not None:
            raise RuntimeError("cloudflared exited early: %s"
                               % " | ".join(found.get("tail") or [])[-300:])
        time.sleep(0.5)
    try:
        proc.terminate()
    except Exception:
        pass
    raise RuntimeError("cloudflared printed no URL within %ds" % timeout_s)


def _teardown(tunnel_proc, backend_handle, port):
    if tunnel_proc is not None:
        try:
            os.killpg(tunnel_proc.pid, signal.SIGTERM)
            tunnel_proc.wait(timeout=8)
        except Exception:
            try:
                os.killpg(tunnel_proc.pid, signal.SIGKILL)
            except Exception:
                pass
    if backend_handle is not None:
        try:
            backend_handle.stop()
        except Exception:
            pass
    try:
        stop_comfy()
    except Exception:
        pass
    _free_port(port)
    _free_port(8188)


# --- bring everything up -----------------------------------------------------

studio_live = LiveBox("studio", "03 . studio starting", rows=[
    ("repo", "fetching origin/%s ..." % STUDIO_BRANCH),
], kind="run",
    note="ComfyUI boots once here; the Studio then restarts it only when a job "
         "switches model family (H3 needs --force-fp16, LTX-2.5 runs without it).")

repo_state = _ensure_repo()
ui_state = _build_ui()
sys.path.insert(0, str(REPO_DIR / "server"))
sys.modules.pop("h3_backend", None)
import h3_backend  # noqa: E402

studio_live.set(rows=[("repo", repo_state), ("ui build", ui_state),
                      ("ComfyUI", "starting ...")])
try:
    start_comfy(None, wait_s=900)
    H3["server_family"] = "base"
    comfy_state = "up on 127.0.0.1:8188 (flags are chosen per model on the first job)"
except Exception as exc:
    comfy_state = "prestart failed - jobs retry on demand (%s)" % str(exc)[:180]

studio_port = _pick_port()
backend = h3_backend.start(port=studio_port, hooks={
    "dist_dir": str(REPO_DIR / "dist"),
    "comfy_url": BASE_URL,
    "output_dir": str(OUTPUT_DIR),
    "upload_dir": str(UPLOAD_DIR),
    "make_job": run_job,
    "list_loras": _lora_choices,
    "download_lora": download_lora,
    "log_tail": _log_tail_text,
    "setup_state": setup_state,
    "apply_setup": apply_setup,
    "install_sage": install_sage,
    "stage_upload": _stage_upload,
})

studio_live.set(rows=[
    ("repo", repo_state),
    ("ui build", ui_state),
    ("ComfyUI", comfy_state),
    ("web ui + api", "127.0.0.1:%d" % studio_port),
    ("tunnel", "opening ..."),
])

tunnel_proc, tunnel_url, tunnel_error = None, None, ""
try:
    _ensure_cloudflared()
    tunnel_proc, tunnel_url = _start_tunnel(studio_port)
except Exception as exc:
    tunnel_error = "%s: %s" % (type(exc).__name__, str(exc)[:300])

if tunnel_url:
    studio_live.set(title="03 . studio live", rows=[
        ("repo", repo_state),
        ("ui build", ui_state),
        ("ComfyUI", comfy_state),
        ("web ui + api", "127.0.0.1:%d" % studio_port),
        ("url", '<a href="%s" target="_blank" style="color:#1a5fb4">%s</a>'
         % (tunnel_url, tunnel_url)),
    ], kind="ok",
        note="Open the URL in a new tab — that is the whole Studio (ComfyUI tunneled "
             "through the same host). Stopping this cell closes the tunnel, the web "
             "server and ComfyUI.")
else:
    studio_live.set(title="03 . studio running (tunnel unavailable)", rows=[
        ("repo", repo_state),
        ("ui build", ui_state),
        ("ComfyUI", comfy_state),
        ("web ui + api", "127.0.0.1:%d" % studio_port),
        ("tunnel", tunnel_error or "not started"),
    ], kind="warn",
        note="The web UI and ComfyUI are up; only the public tunnel failed. Stop this "
             "cell and rerun it to retry.")

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    pass
finally:
    _teardown(tunnel_proc, backend, studio_port)

studio_live.finish("03 . studio stopped", rows=[
    ("tunnel", "closed"),
    ("web ui", "stopped"),
    ("ComfyUI", "stopped"),
], kind="idle",
    note="Ports %d and 8188 freed. Rerun this cell to start again." % studio_port)
